# v13 GNN 이진 분류 — 실행 가능본

원본: `AML_GNN_v13_clean (2).ipynb` (13셀, 전 셀 실행 완료본)

```
"""v13 GNN 이진 분류 — 실행 가능본 (AML_GNN_v13_clean (2).ipynb 변환 + 언더샘플링 훅)

원본: /workspace/AML_GNN_v13_clean (2).ipynb  (13셀, 전 셀 실행 완료본)
변환 원칙 — 노트북 코드를 **의미적으로 바꾸지 않는다**. 추가한 것만 아래에 명시한다.

[원본과 동일 (한 글자도 안 바꿈)]
  §1 로드·정제·분할 / §2 공통 64차원 엣지피처 / §3 Track B 106차원 / §4 패턴 역매칭
  §5 MegaConv·EdgeGINe 아키텍처 / §6 평가 프로토콜(듀얼 예산·패턴 재현율·report_row)
  §9 하이브리드 결합(rank01·combine_scores·select_v14)

[추가한 것 — 전부 옵션이며 기본값은 원본 동작]
  1. --stage 분리 + 전처리 캐시. 원본은 매 실행 250초를 다시 쓴다. 캐시는 계산 결과를
     그대로 저장할 뿐 값을 바꾸지 않는다(base 단계에서 원본 코드 그대로 만든 배열을 덤프).
  2. 확률·가중치 저장. 원본은 torch.save 가 0회라 어떤 후속 실험도 재학습을 강제한다.
  3. 언더샘플링 훅 두 가지 — 이것이 이번 실험의 대상이다.
       --us mask  : 그래프는 온전히 두고 **손실 계산에서만** 다수 표본을 제외한다.
                    ei/ea/x/pinv/pair_dst 가 전부 그대로라 메시지 전달 구조가 보존된다.
       --us prune : 학습 구간에서 해당 **행(엣지) 자체를 삭제**한다. 창 수가 줄어 학습이
                    빨라지지만 남은 거래의 1-hop 이웃이 사라진다.
     둘의 차이를 재는 것이 목적이므로 같은 인덱스(같은 subset)를 두 방식에 똑같이 먹인다.
  4. --oop drop : 패턴 밖 라벨1 을 학습에서 제외(행 삭제). 기본값은 keep(원본 동작).
  5. Track B 스윕 → 1칸 고정(--trackb-sweep 로 원복 가능). 근거: 좌표하강 15칸 × 3.05M행
     × 106차원 × 1500트리는 마감 안에 못 돈다. 1칸만 돌리고 그 사실을 표에 명시한다.

주의 — 재현성: MegaConv 가 index_add 를 쓰고 CUDA index_add 는 부동소수점 누적 순서가
비결정적이다. 시드를 고정해도 소수점 아래가 실행마다 흔들린다. 재실행값이 노트북값과
```

In [1]:
import argparse
import gc
import json
import os
import random
import sys
import time
import warnings
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_curve,
    precision_score, recall_score
)

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# 실행 설정 — 터미널 --stage 인자 대신 이 딕셔너리 값을 바꿔서 재실행한다.
#            (뒤의 모든 코드는 ARGS.xxx 형태로 이 값을 그대로 읽으므로 다른 셀은 손댈 필요 없다)
# ══════════════════════════════════════════════════════════════════════════════
CONFIG = dict(
    stage="all",              # base | usidx | trackA | trackB | hybrid | all
    us="none",                # none=원본 / mask=손실 마스킹(그래프 보존) / prune=엣지 삭제
    method="ensemble_cssmc",  # 언더샘플 방식 (usidx 셀이 만들어 둔 이름)
    ratio=30.0,               # 정상:양성 비율 (30 = 30:1)
    oop="keep",               # keep=원본 / drop=패턴 밖 라벨1 학습에서 행 삭제
    seeds="0,1",
    epochs=0,                 # 0 이면 원본값(25)
    patience=0,               # 0 이면 원본값(5)
    tag="base",
    cachedir="/workspace/us_v14/cache",
    outdir="/workspace/artifacts_v13_undersample",
    trackb_sweep=False,       # True 면 원본 좌표하강 15칸 스윕 복원(느림)
    hybrid_a="",              # hybrid 셀에서 쓸 Track A tag
    hybrid_b="base",          # hybrid 셀에서 쓸 Track B tag
)


class _Args:
    def __init__(self, d):
        self.__dict__.update(d)


ARGS = _Args(CONFIG)

CACHE = Path(ARGS.cachedir)
OUT = Path(ARGS.outdir)
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

In [3]:
# §0. 설정 — 원본 cell 2 와 동일 (EPOCHS/PATIENCE/SEED_LIST 만 CLI 로 덮어쓸 수 있음)
# ══════════════════════════════════════════════════════════════════════════════
SMOKE = os.environ.get("AML_V13_SMOKE", "0") == "1"
DATASET = "HI-Small"
DATA_DIRS = [Path("/workspace/IBM_AML_dataset"), Path("IBM_AML_dataset"), Path("."),
             Path.home() / "Downloads" / "IBM_AML_dataset", Path.home() / "Downloads"]
MAX_ROWS = int(os.environ.get("AML_V13_MAX_ROWS", "0")) or (300_000 if SMOKE else None)
ROW_OFFSET = int(os.environ.get("AML_V13_ROW_OFFSET", "-1"))
if ROW_OFFSET < 0:
    ROW_OFFSET = 3_000_000 if SMOKE else 0

TRAIN_FRAC, VAL_FRAC = 0.6, 0.2
TAIL_MIN_FRAC = 0.05

WINDOW_SIZE, CONTEXT_EDGES = 100_000, 100_000
USE_PORTS, USE_MEGA = True, True
NODE_FEAT_DIM = 9
HIDDEN, N_LAYERS, DROPOUT = 256, 2, 0.15
LR, WEIGHT_DECAY = 5e-4, 1e-5
EPOCHS, PATIENCE = (15, 6) if SMOKE else (25, 5)
if ARGS.epochs:
    EPOCHS = ARGS.epochs
if ARGS.patience:
    PATIENCE = ARGS.patience
POS_WEIGHT_CAP, GRAD_CLIP = 5.0, 1.0
SEED_LIST = [int(s) for s in ARGS.seeds.split(",") if s.strip() != ""]
USE_NODE_AUX, AUX_NODE_W = True, 0.6
MSG_DROPOUT = 0.0
LOSS_TYPE, FOCAL_GAMMA = "focal", 1.0
ALERT_BUDGET = 0.001

USE_AMP = True
AMP_DTYPE = (torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
             else torch.float16)
AGG_CLAMP, AGG_CLAMP_FRAC, AMP_HEADROOM_WARN = True, 0.25, 0.25

LGB_N_ESTIMATORS = 500 if SMOKE else 1500
LGB_EARLY_STOP = 50 if SMOKE else 100
LGB_W_GRID = [1.0, 5.0, 15.0] if SMOKE else [1.0, 5.0, 15.0, 30.0, 50.0]
LGB_LR_GRID = [0.03, 0.05, 0.1]
LGB_LEAF_GRID = [31, 63] if SMOKE else [31, 63, 127, 255]
LGB_MCS_GRID = [20, 50] if SMOKE else [20, 50, 200]
LGB_OOP_GRID = [1.0, 3.0, 5.0, 8.0]

HYBRID_W_GRID = [round(x, 3) for x in np.arange(0.0, 1.0001, 0.05)]
HYBRID_PRAUC_TOL = 0.010
TARGET_VAL_PRAUC, TARGET_OUT_RECALL = 0.6400, 20.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 언더샘플 군집 수 — prep9/undersample.py 관례(k-means centroid) 를 그대로 따른다
US_K = 32
US_N_SUBSETS = 3          # 앙상블 부분집합 수 (GNN 은 시드마다 다른 subset 을 먹는다)
US_KLD_TRIES = 4          # CSSMC: 층화무작위 후보 4벌 → KLD 최소 채택

T_START = time.time()

In [4]:
def log(*a):
    print(f"[{time.time()-T_START:8.1f}s]", *a, flush=True)


def find_member(fname):
    for d in DATA_DIRS:
        p = Path(d) / fname
        if p.exists():
            return p
    raise FileNotFoundError(f"'{fname}' 파일을 찾을 수 없습니다.")


def stack_cols(frame, cols):
    out = np.empty((len(frame), len(cols)), dtype=np.float32)
    for j, c in enumerate(cols):
        out[:, j] = frame[c].to_numpy(dtype=np.float32)
    return out

## STAGE base — 원본 노트북 cell 3~6 (로드·정제·분할·64차원 피처·Track B 106차원·패턴 역매칭)

In [5]:
# STAGE base — 원본 cell 3~6 을 그대로 실행하고 결과 배열만 캐시에 덤프한다.
#              (아래 본문은 노트북 원문. 함수로 감싸느라 들여쓰기만 들어갔다.)
# ══════════════════════════════════════════════════════════════════════════════
def build_cache():
    global EPS, CLIP_Z, SCALE_FLOOR, TSEC

    # ---------- 원본 노트북 cell 3 ----------

    # §1. 데이터 로드 · 정제 · 분할
    def find_member(fname):
        for d in DATA_DIRS:
            p = Path(d) / fname
            if p.exists():
                return p
        raise FileNotFoundError(f"'{fname}' 파일을 찾을 수 없습니다.")


    def load_transactions(ds, nrows=None, skip=0):
        p = find_member(f"{ds}_Trans.csv")
        d = pd.read_csv(
            p, nrows=nrows, skiprows=(range(1, skip + 1) if skip else None),
            dtype={"From Bank": str, "Account": str, "To Bank": str, "Account.1": str,
                   "Receiving Currency": "category", "Payment Currency": "category",
                   "Payment Format": "category", "Is Laundering": np.int8}
        )
        d["Timestamp"] = pd.to_datetime(d["Timestamp"], format="%Y/%m/%d %H:%M")
        return d


    def clean_frame(raw):
        n_raw = len(raw)
        d = raw.drop_duplicates()
        n_dup = n_raw - len(d)
        d = d.sort_values("Timestamp", kind="mergesort").reset_index(drop=True)
        daily = d["Timestamp"].dt.floor("D").value_counts().sort_index()
        good = daily[daily >= daily.median() * TAIL_MIN_FRAC].index
        day = d["Timestamp"].dt.floor("D")
        keep = (day >= good.min()) & (day <= good.max())
        d = d[keep].reset_index(drop=True)
        print(f"중복 제거 {n_dup:,}건 / 꼬리 절단 {int((~keep).sum()):,}건 / "
              f"최종 {len(d):,}건 (세탁률 {d['Is Laundering'].mean()*100:.4f}%)")
        return d


    _t0 = time.time()
    df = clean_frame(load_transactions(DATASET, MAX_ROWS, ROW_OFFSET))
    n = len(df)
    df["order_idx"] = np.arange(n, dtype=np.int64)

    n_tr = int(n * TRAIN_FRAC)
    n_va = int(n * (TRAIN_FRAC + VAL_FRAC))
    _split = np.full(n, "test", dtype=object)
    _split[:n_tr] = "train"
    _split[n_tr:n_va] = "val"
    df["split"] = _split
    is_train = df["split"].eq("train").to_numpy()

    _src_key = df["From Bank"].astype(str) + "_" + df["Account"].astype(str)
    _dst_key = df["To Bank"].astype(str) + "_" + df["Account.1"].astype(str)
    _codes, _uniq = pd.factorize(pd.concat([_src_key, _dst_key], ignore_index=True))
    df["src_id"] = _codes[:n].astype(np.int64)
    df["dst_id"] = _codes[n:].astype(np.int64)
    NUM_ACCOUNTS = len(_uniq)
    df["self_loop"] = (df["src_id"] == df["dst_id"]).astype(np.float32)
    del _src_key, _dst_key, _codes, _uniq, _split

    print(f"분할: train {n_tr:,} / val {n_va-n_tr:,} / test {n-n_va:,} | "
          f"고유 계좌 {NUM_ACCOUNTS:,}개 [{time.time()-_t0:.1f}s]")



    # ---------- 원본 노트북 cell 4 ----------

    # §2. 공통 피처 엔지니어링
    def add_base_features(d, is_tr):
        def per_currency_z(amount_col, currency_col):
            la = np.log1p(d[amount_col].clip(lower=0).to_numpy(dtype=np.float64))
            cur = d[currency_col].astype(str)
            s = pd.DataFrame({"cur": cur, "la": la})
            st = s[is_tr].groupby("cur")["la"].agg(["mean", "std"])
            st["std"] = st["std"].replace(0, np.nan)
            g_mean, g_std = s.loc[is_tr, "la"].mean(), s.loc[is_tr, "la"].std()
            m_ = cur.map(st["mean"]).astype(np.float64).fillna(g_mean).to_numpy()
            sd = cur.map(st["std"]).astype(np.float64).fillna(g_std).to_numpy()
            return ((la - m_) / sd).astype(np.float32)

        d["amt_paid_z"] = per_currency_z("Amount Paid", "Payment Currency")
        d["amt_recv_z"] = per_currency_z("Amount Received", "Receiving Currency")
        d["cur_mismatch"] = (d["Payment Currency"].astype(str) != d["Receiving Currency"].astype(str)).astype(np.float32)
        d["amt_neq"] = (d["Amount Paid"] != d["Amount Received"]).astype(np.float32)

        def onehot(col, prefix, max_cats=20):
            cs = d.loc[is_tr, col].astype(str).value_counts().index[:max_cats].tolist()
            v = d[col].astype(str)
            return pd.DataFrame({f"{prefix}_{c}": (v == c).astype(np.float32) for c in cs}, index=d.index)

        pf_oh = onehot("Payment Format", "pf")
        pc_oh = onehot("Payment Currency", "pc")
        rc_oh = onehot("Receiving Currency", "rc")
        d = pd.concat([d, pf_oh, pc_oh, rc_oh], axis=1)

        nloc, logN = len(d), np.log1p(len(d))
        for role, key in [("src", "src_id"), ("dst", "dst_id")]:
            g = d.groupby(key)["order_idx"]
            d[f"{role}_seq"] = (np.log1p(g.cumcount()) / logN).astype(np.float32)
            prev = g.shift(1)
            d[f"{role}_first"] = prev.isna().astype(np.float32)
            d[f"{role}_gap"] = (np.log1p((d["order_idx"] - prev).fillna(nloc)) / logN).astype(np.float32)

        base = (["amt_paid_z", "amt_recv_z", "cur_mismatch", "amt_neq", "self_loop",
                 "src_seq", "src_first", "src_gap", "dst_seq", "dst_first", "dst_gap"]
                + list(pf_oh.columns) + list(pc_oh.columns) + list(rc_oh.columns))
        return d, base


    def prior_stats(acct, tsec, windows):
        K = np.int64(10_000_000)
        keys = acct.astype(np.int64) * K + tsec
        skeys = np.sort(keys, kind="stable")
        pos = np.searchsorted(skeys, keys, side="left")
        prev_idx = pos - 1
        ok = prev_idx >= 0
        prev_key = np.where(ok, skeys[np.clip(prev_idx, 0, None)], -1)
        same = ok & ((prev_key // K) == acct.astype(np.int64))
        dt = np.where(same, tsec - (prev_key % K), -1).astype(np.float64)
        out = {"dt": dt, "is_first": (~same).astype(np.float32)}
        for w in windows:
            lo = np.searchsorted(skeys, acct.astype(np.int64) * K + np.maximum(tsec - w, 0), side="left")
            out[f"cnt{w}"] = (pos - lo).astype(np.float32)
        return out


    TIME_WIN = [3600, 86400]
    _ts_abs = df["Timestamp"].to_numpy().astype("datetime64[s]").astype(np.int64)
    TSEC = (_ts_abs - _ts_abs.min())
    del _ts_abs


    def add_time_features(d):
        nloc = len(d)
        LOGT, LOGC = np.log1p(86400.0 * 14), np.log(50.0)
        tfeats = []
        src_np, dst_np = d["src_id"].to_numpy(), d["dst_id"].to_numpy()
        pair_id = pd.factorize(pd.Series(src_np * np.int64(1 << 21) + dst_np))[0].astype(np.int64)
        both = prior_stats(np.concatenate([src_np, dst_np]), np.concatenate([TSEC, TSEC]), TIME_WIN)

        for role, sl in [("src", slice(0, nloc)), ("dst", slice(nloc, 2 * nloc))]:
            dd = both["dt"][sl]
            d[f"{role}_dt"] = np.where(dd < 0, 1.0, np.log1p(np.maximum(dd, 0)) / LOGT).astype(np.float32)
            d[f"{role}_dt_first"] = both["is_first"][sl]
            tfeats += [f"{role}_dt", f"{role}_dt_first"]
            for w in TIME_WIN:
                d[f"{role}_cnt{w}"] = (np.log1p(both[f"cnt{w}"][sl]) / LOGC).astype(np.float32)
                tfeats.append(f"{role}_cnt{w}")

        out_s, in_s = prior_stats(src_np, TSEC, TIME_WIN), prior_stats(dst_np, TSEC, TIME_WIN)
        for w in TIME_WIN:
            d[f"out_cnt{w}"] = (np.log1p(out_s[f"cnt{w}"]) / LOGC).astype(np.float32)
            d[f"in_cnt{w}"] = (np.log1p(in_s[f"cnt{w}"]) / LOGC).astype(np.float32)
            tfeats += [f"out_cnt{w}", f"in_cnt{w}"]

        pr = prior_stats(pair_id, TSEC, TIME_WIN)
        d["pair_dt"] = np.where(pr["dt"] < 0, 1.0, np.log1p(np.maximum(pr["dt"], 0)) / LOGT).astype(np.float32)
        d["pair_first"] = pr["is_first"]
        tfeats += ["pair_dt", "pair_first"]
        for w in TIME_WIN:
            d[f"pair_cnt{w}"] = (np.log1p(pr[f"cnt{w}"]) / LOGC).astype(np.float32)
            tfeats.append(f"pair_cnt{w}")

        raw = dict(src_dt=both["dt"][:nloc], dst_dt=both["dt"][nloc:],
                   src_first=both["is_first"][:nloc], dst_first=both["is_first"][nloc:],
                   out1h=out_s["cnt3600"], out24h=out_s["cnt86400"],
                   in1h=in_s["cnt3600"], in24h=in_s["cnt86400"],
                   pair1h=pr["cnt3600"], pair24h=pr["cnt86400"],
                   pair_dt=pr["dt"], pair_id=pair_id)
        return d, tfeats, raw


    df, BASE_FEATURES = add_base_features(df, is_train)
    df, TIME_FEATURES, RAW_T = add_time_features(df)
    EDGE_FEATURES = BASE_FEATURES + TIME_FEATURES
    print(f"공통 엣지 피처: {len(EDGE_FEATURES)}차원 완성")



    # ---------- 원본 노트북 cell 5 ----------

    # §3. [v13 신규] Track B 행동 이상치 + Structuring 6종 + [v14] 오염도·고차적률·차수·역방향 18종
    class PastIndex:
        def __init__(self, keys, tsec):
            n_ = len(keys)
            order = np.lexsort((tsec, keys))
            k, t = np.asarray(keys)[order], np.asarray(tsec)[order]
            new_g = np.empty(n_, dtype=bool)
            new_g[0] = True
            new_g[1:] = k[1:] != k[:-1]
            new_b = new_g.copy()
            new_b[1:] |= t[1:] != t[:-1]
            bstart = np.flatnonzero(new_b)
            self.n, self.order, self.k = n_, order, k
            self.bid = np.cumsum(new_b) - 1
            self.prev = np.maximum(bstart - 1, 0)
            self.first = new_g[bstart]
            incc = pd.Series(np.zeros(n_)).groupby(k, sort=False).cumcount().to_numpy() + 1
            self.cnt = self._take(incc.astype(np.float64))

        def _take(self, arr_sorted):
            out = np.empty(self.n, dtype=np.float64)
            out[self.order] = np.where(self.first[self.bid], 0.0, arr_sorted[self.prev][self.bid])
            return out

        def sum(self, vals):
            v = np.asarray(vals, dtype=np.float64)[self.order]
            return self._take(pd.Series(v).groupby(self.k, sort=False).cumsum().to_numpy())

        def min(self, vals):
            v = np.asarray(vals, dtype=np.float64)[self.order]
            return self._take(pd.Series(v).groupby(self.k, sort=False).cummin().to_numpy())

        def moments(self, vals, valid=None):
            v = np.asarray(vals, dtype=np.float64)
            if valid is None:
                nval = self.cnt
                sel = np.ones(len(v), dtype=bool)
            else:
                m = np.asarray(valid, dtype=np.float64)
                nval = self.sum(m)
                sel = m > 0
            c0 = float(v[sel].mean()) if sel.any() else 0.0
            vc = (v - c0) * (sel.astype(np.float64))
            s1, s2 = self.sum(vc), self.sum(vc * vc)
            c = np.maximum(nval, 1.0)
            mc = s1 / c
            return nval, mc + c0, np.sqrt(np.maximum(s2 / c - mc * mc, 0.0))

        # ── [v14 · 3단계] 과거 전용 4차 중심적률 → mean / std / skew / kurtosis ──
        def moments4(self, vals):
            v = np.asarray(vals, dtype=np.float64)
            n = self.cnt
            c0 = float(v.mean())
            u = v - c0
            S1, S2, S3, S4 = self.sum(u), self.sum(u * u), self.sum(u ** 3), self.sum(u ** 4)
            c = np.maximum(n, 1.0)
            M1 = S1 / c
            m2 = np.maximum(S2 / c - M1 * M1, 0.0)
            m3 = S3 / c - 3 * M1 * (S2 / c) + 2 * M1 ** 3
            m4 = S4 / c - 4 * M1 * (S3 / c) + 6 * M1 * M1 * (S2 / c) - 3 * M1 ** 4
            sd = np.sqrt(m2)
            skew = np.where(n >= 3, m3 / np.maximum(sd ** 3, EPS), 0.0)
            kurt = np.where(n >= 4, m4 / np.maximum(m2 * m2, EPS) - 3.0, 0.0)
            return n, M1 + c0, sd, skew, kurt


    EPS = 1e-5
    CLIP_Z = 30.0
    SCALE_FLOOR = 0.05


    # ──────────────────────────────────────────────────────────────────────────
    # [v14 · 4단계 전제] 과거 이력 기반 계좌 차수(deg) — 미래 엣지를 보지 않는 누수-프리 정의
    # ──────────────────────────────────────────────────────────────────────────
    def build_past_degree(d):
        """각 거래 '직전'까지 관측된 서로 다른 거래상대 수.

        무방향 페어 (a,b) 는 '최초 등장 행'에서 양 끝 계좌의 distinct 카운트를 +1 시킨다.
        (계좌, 최초등장행) 이벤트를 정렬해 두고 이분탐색으로 과거 개수를 센다.
        deg == 0 이면 사전 이력이 전혀 없는 신규 계좌.
        """
        s = d["src_id"].to_numpy().astype(np.int64)
        t = d["dst_id"].to_numpy().astype(np.int64)
        n_all = len(d)
        n_acct = int(max(s.max(), t.max())) + 1
        row = np.arange(n_all, dtype=np.int64)

        lo_a, hi_a = np.minimum(s, t), np.maximum(s, t)
        pair = lo_a * np.int64(n_acct) + hi_a
        order = np.argsort(pair, kind="stable")          # stable → 페어별 최초 행이 앞에 온다
        ps = pair[order]
        fp = np.flatnonzero(np.r_[True, ps[1:] != ps[:-1]])
        first_row = order[fp]
        up = ps[fp]
        ua, ub = up // np.int64(n_acct), up % np.int64(n_acct)
        keep = ua != ub                                   # self-loop 은 상대 수에 포함하지 않음
        ev_acct = np.concatenate([ua[keep], ub[keep]])
        ev_row = np.concatenate([first_row[keep], first_row[keep]])

        SHIFT = np.int64(1) << np.int64(int(np.ceil(np.log2(max(n_all, 2)))) + 1)
        ev_key = np.sort(ev_acct * SHIFT + ev_row)

        def prior(acct):
            base = acct * SHIFT
            return (np.searchsorted(ev_key, base + row, side="left")
                    - np.searchsorted(ev_key, base, side="left")).astype(np.float32)

        return prior(s), prior(t)


    # ──────────────────────────────────────────────────────────────────────────
    # [v14 · 2단계] 계좌 오염도(taint) — 라벨의 전이성을 직접 피처로 인코딩
    #   seed 는 train 구간 양성만. val/test 구간은 조회만 하므로 누수 없음.
    #   PastIndex 는 §3 에서 이미 만든 것을 재사용한다(추가 정렬 비용 0).
    # ──────────────────────────────────────────────────────────────────────────
    def build_taint_features(pi_map, src, dst, amt, y, n_tr, n_all):
        seed = np.zeros(n_all, dtype=np.float64)
        seed[:n_tr] = (y[:n_tr] > 0).astype(np.float64)
        seed_amt = seed * amt
        out = {}
        for role, _acct in (("src", src), ("dst", dst)):
            pi = pi_map[role]
            cnt_t = pi.sum(seed)
            amt_t = pi.sum(seed_amt)
            n_prior = np.maximum(pi.cnt, 1.0)
            out[f"tb_taint_cnt_{role}"] = np.log1p(cnt_t)
            out[f"tb_taint_amt_{role}"] = np.log1p(amt_t) / np.log(1e6)
            out[f"tb_taint_rate_{role}"] = cnt_t / n_prior
        out["tb_taint_either"] = np.maximum(out["tb_taint_cnt_src"], out["tb_taint_cnt_dst"])
        out["tb_taint_both"] = np.minimum(out["tb_taint_cnt_src"], out["tb_taint_cnt_dst"])
        return out


    def build_tabular_features_v13(d, raw, tsec, is_tr):
        names = []
        src, dst = d["src_id"].to_numpy(), d["dst_id"].to_numpy()
        paid = d["Amount Paid"].to_numpy(dtype=np.float64)
        recv = d["Amount Received"].to_numpy(dtype=np.float64)
        lpaid, lrecv = np.log1p(np.maximum(paid, 0)), np.log1p(np.maximum(recv, 0))
        nloc = len(d)

        def put(name, arr):
            d[name] = np.asarray(arr, dtype=np.float32)
            names.append(name)

        PI = {"src": PastIndex(src, tsec), "dst": PastIndex(dst, tsec)}

        # 1) 기본 Amount Z-Score
        for role, amt, lamt in [("src", paid, lpaid), ("dst", recv, lrecv)]:
            pi = PI[role]
            cnt, mean, std = pi.moments(amt)
            denom = np.maximum(std, SCALE_FLOOR * np.maximum(np.abs(mean), 1.0)) + EPS
            put(f"tb_amt_z_{role}", np.clip(np.where(cnt > 0, (amt - mean) / denom, 0.0), -CLIP_Z, CLIP_Z))
            _, lmean, lstd = pi.moments(lamt)
            ldenom = np.maximum(lstd, SCALE_FLOOR * np.maximum(np.abs(lmean), 1.0)) + EPS
            put(f"tb_lamt_z_{role}", np.clip(np.where(cnt > 0, (lamt - lmean) / ldenom, 0.0), -CLIP_Z, CLIP_Z))
            if role == "src":
                put("tb_amt_ratio_src", np.where(cnt > 0, np.log1p(amt / (mean + 1.0)), 0.0))

        # 2) Velocity Ratio
        for tag, c1, c24 in [("src", raw["out1h"], raw["out24h"]),
                             ("dst", raw["in1h"], raw["in24h"]),
                             ("pair", raw["pair1h"], raw["pair24h"])]:
            vel = c1.astype(np.float64) / (c24.astype(np.float64) / 24.0 + EPS)
            put(f"tb_vel_{tag}", np.log1p(np.clip(vel, 0, 1e6)))

        # 3) Δt anomaly
        for role, dt_raw in [("src", raw["src_dt"]), ("dst", raw["dst_dt"])]:
            valid = (dt_raw >= 0)
            dt = np.where(valid, dt_raw, 0.0)
            nval, mean, std = PI[role].moments(dt, valid=valid.astype(np.float64))
            z = (dt - mean) / (std + SCALE_FLOOR * np.maximum(mean, 1.0) + EPS)
            put(f"tb_dt_z_{role}", np.clip(np.where(nval > 1, z, 0.0), -CLIP_Z, CLIP_Z))
            put(f"tb_dt_ratio_{role}", np.where(nval > 0, np.log1p(dt) / (np.log1p(mean) + 1.0), 0.0))

        # 4) 포트 및 잔액 프록시
        pair_cnt = PastIndex(raw["pair_id"], tsec).cnt
        put("tb_port_rank", np.log1p(pair_cnt) / np.log(20.0))
        put("tb_pair_share", pair_cnt / (PI["src"].cnt + 1.0))

        cc = pd.factorize(
            pd.concat([d["Payment Currency"].astype(str), d["Receiving Currency"].astype(str)], ignore_index=True)
        )[0]
        n_cur = np.int64(cc.max() + 1)
        ev_acct = np.empty(2 * nloc, dtype=np.int64)
        ev_val = np.empty(2 * nloc, dtype=np.float64)
        ev_acct[0::2] = src * n_cur + cc[:nloc]
        ev_acct[1::2] = dst * n_cur + cc[nloc:]
        ev_val[0::2], ev_val[1::2] = -paid, recv
        flow = PastIndex(ev_acct, np.repeat(tsec, 2)).sum(ev_val)
        bal_src, bal_dst = flow[0::2], flow[1::2]
        for role, bal, amt in [("src", bal_src, paid), ("dst", bal_dst, recv)]:
            put(f"tb_bal_{role}", np.sign(bal) * np.log1p(np.abs(bal)) / np.log(1e6))
            put(f"tb_bal_ratio_{role}", np.log1p(amt / (np.abs(bal) + 1.0)))
        del ev_acct, ev_val, flow

        # ──────────────────────────────────────────────────────────────────────────
        # 🌟 [v13 4-Way Breakthrough 신규 피처 6종]
        # ──────────────────────────────────────────────────────────────────────────
        put("tb_structuring_10k", np.exp(-((paid - 9800.0) ** 2) / (2.0 * (300.0 ** 2))))
        put("tb_structuring_50k", np.exp(-((paid - 49000.0) ** 2) / (2.0 * (1000.0 ** 2))))

        acct_first_tsec = PI["src"].min(tsec)
        acct_age_days = np.maximum(tsec - acct_first_tsec, 0.0) / 86400.0
        prior_c = PI["src"].cnt
        put("tb_dormant_burst", (np.log1p(paid) / (prior_c + 1.0)) * np.log1p(acct_age_days + 1.0))

        put("tb_round_avoidance", ((paid % 100.0 > 0) & ((paid % 1000.0) >= 800.0)).astype(np.float32))

        is_night = (d["Timestamp"].dt.hour < 6).to_numpy(dtype=np.float32)
        is_cash_cheque = d["Payment Format"].astype(str).isin(["Cash", "Cheque", "Reinvestment"]).to_numpy(dtype=np.float32)
        put("tb_night_cash_burst", np.maximum(is_night, is_cash_cheque))

        z_amt = np.maximum(d["tb_amt_z_src"].to_numpy(), 0.0)
        z_vel = d["tb_vel_src"].to_numpy()
        struct_score = d["tb_structuring_10k"].to_numpy()
        put("tb_iso_composite", np.log1p(z_amt + z_vel + struct_score * 5.0))

        # ══════════════════════════════════════════════════════════════════════════
        # 🌟 [v14] 패턴 밖(전이 라벨) 재현율 개선용 신규 18종
        # ══════════════════════════════════════════════════════════════════════════

        # ── 2단계 · 계좌 오염도 8종 ────────────────────────────────────────────────
        # AMLworld 라벨은 전이적이다: 오염된 계좌의 하류 결제가 평범해 보여도 양성이 된다.
        # "이 계좌가 과거에 알려진 세탁 거래에 몇 번 닿았는가"를 직접 준다.
        y_all = d["Is Laundering"].to_numpy().astype(np.float64)
        taint = build_taint_features(PI, src, dst, paid, y_all, n_tr, nloc)
        for _k in ("tb_taint_cnt_src", "tb_taint_cnt_dst",
                   "tb_taint_amt_src", "tb_taint_amt_dst",
                   "tb_taint_rate_src", "tb_taint_rate_dst",
                   "tb_taint_either", "tb_taint_both"):
            put(_k, taint[_k])
        del taint

        # ── 3단계 · 과거 금액 분포의 고차 적률 4종 ────────────────────────────────
        # 평균·표준편차만으로는 "가끔 튀는" 계좌를 못 잡는다. 왜도/첨도가 그 꼬리를 잡는다.
        for role, lamt in (("src", lpaid), ("dst", lrecv)):
            _n, _m, _sd, _skew, _kurt = PI[role].moments4(lamt)
            put(f"tb_skew_amt_{role}", np.clip(_skew, -CLIP_Z, CLIP_Z))
            put(f"tb_kurt_amt_{role}", np.clip(_kurt, -CLIP_Z, CLIP_Z))

        # ── 4단계 · 과거 이력 기반 차수 4종 ───────────────────────────────────────
        # 게이팅용이 아니라 '모델 피처'로 등록한다(하드게이팅은 사용하지 않음).
        _deg_s, _deg_d = build_past_degree(d)
        d["deg_src"], d["deg_dst"] = _deg_s, _deg_d
        _deg_tot = _deg_s.astype(np.float64) + _deg_d.astype(np.float64)
        put("tb_deg_src", np.log1p(_deg_s))
        put("tb_deg_dst", np.log1p(_deg_d))
        put("tb_deg_total", np.log1p(_deg_tot))
        put("tb_deg_ratio", (_deg_s.astype(np.float64) - _deg_d.astype(np.float64)) / np.maximum(_deg_tot, 1.0))

        # ── 5단계 · 역방향(2홉 왕복) 페어 이력 2종 ────────────────────────────────
        # 방향을 구분해야 하므로 정방향/역방향 키를 '함께' factorize 한 뒤 절반씩 분리한다.
        # 되돌림 송금(A→B 이후 B→A)은 계층화(layering)의 대표 흔적이다.
        _shift = np.int64(1) << np.int64(21)
        _fwd_key = src.astype(np.int64) * _shift + dst.astype(np.int64)
        _rev_key = dst.astype(np.int64) * _shift + src.astype(np.int64)
        _codes = pd.factorize(pd.Series(np.concatenate([_fwd_key, _rev_key])))[0].astype(np.int64)
        _fwd_id, _rev_id = _codes[:nloc], _codes[nloc:]
        # [정방향 이벤트 ; 역방향 질의] 를 한 PastIndex 에 넣고, 정방향에만 1 을 주어 합산한다.
        # → 질의 위치(뒤쪽 절반)에는 '같은 키의 과거 정방향 발생 횟수'만 누적된다.
        _pi_rev = PastIndex(np.concatenate([_fwd_id, _rev_id]), np.concatenate([tsec, tsec]))
        _is_fwd = np.concatenate([np.ones(nloc, dtype=np.float64), np.zeros(nloc, dtype=np.float64)])
        _rev_cnt = _pi_rev.sum(_is_fwd)[nloc:]
        put("tb_rev_cnt", np.log1p(_rev_cnt))
        put("tb_rev_flag", (_rev_cnt > 0).astype(np.float32))
        del _pi_rev, _codes, _fwd_key, _rev_key, _fwd_id, _rev_id, _is_fwd

        return d, names


    _t0 = time.time()
    df, TAB_EXTRA_V13 = build_tabular_features_v13(df, RAW_T, TSEC, is_train)
    TAB_FEATURES = EDGE_FEATURES + TAB_EXTRA_V13
    print(f"[Track B v14] 신규 피처 포함 총 {len(TAB_FEATURES)}차원 "
          f"(공통 {len(EDGE_FEATURES)} + 행동이상치 {len(TAB_EXTRA_V13)}) [{time.time()-_t0:.1f}s]")

    # 신규 피처 건전성 점검 (NaN/inf 가 하나라도 있으면 즉시 실패시킨다)
    _V14_NEW = [c for c in TAB_EXTRA_V13 if c.startswith(("tb_taint", "tb_skew", "tb_kurt", "tb_deg", "tb_rev"))]
    for _c in _V14_NEW:
        _v = df[_c].to_numpy()
        assert np.isfinite(_v).all(), f"{_c} 에 NaN/inf 발생"
    print(f"[v14 신규 {len(_V14_NEW)}종] 유한성 OK | "
          f"오염도 점등(val) {float((df['tb_taint_either'].to_numpy()[n_tr:n_va] > 0).mean())*100:.2f}% | "
          f"역방향 점등(val) {float(df['tb_rev_flag'].to_numpy()[n_tr:n_va].mean())*100:.2f}%")



    # ---------- 원본 노트북 cell 6 ----------

    # §4. 패턴 블록 역매칭
    KNOWN_TYPES = ["SCATTER-GATHER", "GATHER-SCATTER", "FAN-OUT", "FAN-IN",
                   "BIPARTITE", "STACK", "CYCLE", "RANDOM"]


    def block_type(name):
        u = name.upper()
        for k in KNOWN_TYPES:
            if k in u:
                return k
        return name.split(":")[0].strip() or "UNKNOWN"


    with open(find_member(f"{DATASET}_Patterns.txt"), "rb") as _f:
        _pat_lines = _f.read().decode("utf-8", errors="ignore").splitlines()

    pattern_blocks, _cur = [], None
    for _ln in _pat_lines:
        if _ln.startswith("BEGIN LAUNDERING ATTEMPT"):
            _cur = {"name": _ln.split("-", 1)[-1].strip(), "rows": []}
        elif _ln.startswith("END LAUNDERING ATTEMPT"):
            if _cur and _cur["rows"]:
                pattern_blocks.append(_cur)
            _cur = None
        elif _cur is not None and _ln.strip():
            _parts = _ln.split(",")
            if len(_parts) >= 11:
                _cur["rows"].append(_parts)

    _amt = df["Amount Paid"].map(lambda v: f"{v:.2f}")
    _key = (df["Timestamp"].dt.strftime("%Y/%m/%d %H:%M") + "|" +
            df["From Bank"].astype(str) + "|" + df["Account"].astype(str) + "|" +
            df["To Bank"].astype(str) + "|" + df["Account.1"].astype(str) + "|" +
            df["Payment Format"].astype(str) + "|" + _amt)

    _lut = defaultdict(deque)
    for _i, _k in enumerate(_key.to_numpy()):
        _lut[_k].append(_i)

    PAT_BLOCKS, PAT_MASK = [], np.zeros(n, dtype=bool)
    for _blk in pattern_blocks:
        _idxs = []
        for _r in _blk["rows"]:
            k = (_r[0].strip() + "|" + _r[1].strip() + "|" + _r[2].strip() + "|" +
                 _r[3].strip() + "|" + _r[4].strip() + "|" + _r[9].strip() + "|" + f"{float(_r[7]):.2f}")
            q = _lut.get(k)
            if q:
                gi = q.popleft()
                _idxs.append(gi)
                PAT_MASK[gi] = True
        if _idxs:
            _idxs.sort()
            PAT_BLOCKS.append(dict(type=block_type(_blk["name"]), idxs=np.array(_idxs, dtype=np.int64)))

    del _lut, _key, _amt

    Y = df["Is Laundering"].to_numpy().astype(np.float32)
    val_y, test_y = Y[n_tr:n_va], Y[n_va:]
    SEG_RANGE = {"val": (n_tr, n_va), "test": (n_va, n)}
    OOP_TIE_TOL = 100.0 / max(int((~PAT_MASK[n_tr:n_va][Y[n_tr:n_va] > 0]).sum()), 1)

    print(f"패턴 매칭 완료: Val 양성 {int((Y[n_tr:n_va]>0).sum())}건 / "
          f"Test 양성 {int((Y[n_va:]>0).sum())}건 (동률허용폭 {OOP_TIE_TOL:.2f}%p)")


    # ── 캐시 덤프 (계산 결과를 그대로 저장할 뿐 값은 바꾸지 않는다) ──
    log("캐시 덤프 시작")
    np.save(CACHE / "SRC.npy", df["src_id"].to_numpy().astype(np.int64))
    np.save(CACHE / "DST.npy", df["dst_id"].to_numpy().astype(np.int64))
    np.save(CACHE / "EF.npy", stack_cols(df, EDGE_FEATURES))
    np.save(CACHE / "A_PAID.npy", df["amt_paid_z"].to_numpy().astype(np.float32))
    np.save(CACHE / "A_RECV.npy", df["amt_recv_z"].to_numpy().astype(np.float32))
    np.save(CACHE / "Y.npy", Y)
    np.save(CACHE / "PAT_MASK.npy", PAT_MASK)
    np.save(CACHE / "XTAB.npy", stack_cols(df, TAB_FEATURES))
    np.save(CACHE / "SRC_SEQ.npy", df["src_seq"].to_numpy().astype(np.float32))
    np.save(CACHE / "DST_SEQ.npy", df["dst_seq"].to_numpy().astype(np.float32))

    _off, _cat, _typ = [0], [], []
    for b in PAT_BLOCKS:
        _cat.append(b["idxs"])
        _off.append(_off[-1] + len(b["idxs"]))
        _typ.append(b["type"])
    np.savez(CACHE / "blocks.npz",
             idxs=np.concatenate(_cat) if _cat else np.zeros(0, np.int64),
             off=np.array(_off, np.int64), typ=np.array(_typ, dtype=object))

    (CACHE / "meta.json").write_text(json.dumps({
        "n": int(n), "n_tr": int(n_tr), "n_va": int(n_va),
        "NUM_ACCOUNTS": int(NUM_ACCOUNTS),
        "EDGE_FEATURES": list(EDGE_FEATURES), "TAB_FEATURES": list(TAB_FEATURES),
        "n_pos_train": int((Y[:n_tr] > 0).sum()),
        "n_pos_train_in": int(((Y[:n_tr] > 0) & PAT_MASK[:n_tr]).sum()),
        "n_pos_train_out": int(((Y[:n_tr] > 0) & ~PAT_MASK[:n_tr]).sum()),
        "n_blocks": len(PAT_BLOCKS),
        "prep_seconds": round(time.time() - T_START, 1),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    log(f"캐시 저장 완료 → {CACHE}")


# ══════════════════════════════════════════════════════════════════════════════
# 캐시 로드 — 이후 모든 단계는 df 없이 배열만 쓴다
# ══════════════════════════════════════════════════════════════════════════════

In [6]:
def load_cache():
    g = globals()
    meta = json.loads((CACHE / "meta.json").read_text(encoding="utf-8"))
    g["META"] = meta
    g["n"], g["n_tr"], g["n_va"] = meta["n"], meta["n_tr"], meta["n_va"]
    g["NUM_ACCOUNTS"] = meta["NUM_ACCOUNTS"]
    g["EDGE_FEATURES"] = meta["EDGE_FEATURES"]
    g["TAB_FEATURES"] = meta["TAB_FEATURES"]
    g["SRC"] = np.load(CACHE / "SRC.npy")
    g["DST"] = np.load(CACHE / "DST.npy")
    g["EF"] = np.load(CACHE / "EF.npy")
    g["A_PAID"] = np.load(CACHE / "A_PAID.npy")
    g["A_RECV"] = np.load(CACHE / "A_RECV.npy")
    g["Y"] = np.load(CACHE / "Y.npy")
    g["PAT_MASK"] = np.load(CACHE / "PAT_MASK.npy")
    z = np.load(CACHE / "blocks.npz", allow_pickle=True)
    off, idxs, typ = z["off"], z["idxs"], z["typ"]
    g["PAT_BLOCKS"] = [dict(type=str(typ[i]), idxs=idxs[off[i]:off[i + 1]])
                       for i in range(len(off) - 1)]
    g["val_y"], g["test_y"] = g["Y"][n_tr:n_va], g["Y"][n_va:]
    g["SEG_RANGE"] = {"val": (n_tr, n_va), "test": (n_va, n)}
    g["OOP_TIE_TOL"] = 100.0 / max(
        int((~g["PAT_MASK"][n_tr:n_va][g["Y"][n_tr:n_va] > 0]).sum()), 1)
    g["EDGE_EXTRA"] = 2 + (2 if USE_PORTS else 0)
    g["EDGE_DIM"] = len(EDGE_FEATURES) + g["EDGE_EXTRA"]
    g["NODE_DIM"] = NODE_FEAT_DIM
    log(f"캐시 로드: n={n:,} train={n_tr:,} val={n_va-n_tr:,} test={n-n_va:,} "
        f"| EF {g['EF'].shape} | 블록 {len(g['PAT_BLOCKS'])}")


# ══════════════════════════════════════════════════════════════════════════════
# 학습 범위(scope) — 패턴 밖 라벨1 처리
#   keep : 원본. 패턴 밖 양성도 양성으로 학습한다.
#   drop : 팀 결정. 패턴 밖 양성 '행 자체'를 학습에서 뺀다(라벨0 재라벨이 아니다).
#          평가 구간(val/test)은 절대 건드리지 않는다 — 건드리면 다른 실험과 비교 불가.
# ══════════════════════════════════════════════════════════════════════════════

## 언더샘플링 인덱스 준비 — CSSMC 층화·KLD·k-medoids(CLARA) 등

In [7]:
def train_scope_mask():
    """train 구간에서 '학습에 쓸 행' 불리언 (길이 n_tr)."""
    m = np.ones(n_tr, dtype=bool)
    if ARGS.oop == "drop":
        m &= ~((Y[:n_tr] > 0) & (~PAT_MASK[:n_tr]))
    return m


# ══════════════════════════════════════════════════════════════════════════════
# STAGE usidx — 언더샘플 인덱스 생성
#   방식: prep9/undersample.py 의 fit_clusters/sample_indices 를 그대로 재사용한다.
#         CSSMC(김주미·정여진, J.KIIT 22(5) 2024 §3.1) = 다수 클래스를 군집화한 뒤
#         군집별 층화 '무작위' 추출(center_frac=0.0) → 후보 여러 벌 중 KLD 최소 채택.
#         prep9 기본값 center_frac=0.5 는 중심 근접 우선이라 CSSMC 와 다르다.
#   누수 방지: 군집 중심·표준화 통계는 train 정상 행에서만 적합한다.
# ══════════════════════════════════════════════════════════════════════════════
def kld_reference(pop, bins=20):
    """모집단 쪽 분위 경계와 히스토그램을 **한 번만** 계산해 둔다.

    후보마다 다시 계산하면 3.04M × 64차원 히스토그램을 24번 돌게 되어
    KLD 계산이 언더샘플링 자체보다 비싸진다. 모집단은 변하지 않으므로 캐시한다.
    """
    ref = []
    for j in range(pop.shape[1]):
        edges = np.unique(np.quantile(pop[:, j], np.linspace(0, 1, bins + 1)))
        if len(edges) < 3:
            ref.append(None)
            continue
        p = np.histogram(pop[:, j], bins=edges)[0].astype(np.float64)
        p = (p + 1e-9) / (p.sum() + 1e-9 * len(p))
        ref.append((edges, p))
    return ref


def _kld_dim(ref, sub):
    """차원별 KLD 평균. 모집단 분위 경계로 이산화한 뒤 KL(sub || pop)."""
    tot = 0.0
    for j, r in enumerate(ref):
        if r is None:
            continue
        edges, p = r
        q = np.histogram(sub[:, j], bins=edges)[0].astype(np.float64)
        q = (q + 1e-9) / (q.sum() + 1e-9 * len(q))
        tot += float((q * np.log(q / p)).sum())
    return tot / max(len(ref), 1)


def build_us_indices():
    sys.path.insert(0, "/workspace")
    from prep9.undersample import fit_clusters, sample_indices   # 재사용 (수정 0줄)

    scope = train_scope_mask()
    ytr = Y[:n_tr]
    pos_idx = np.flatnonzero((ytr > 0) & scope)
    neg_idx = np.flatnonzero((ytr <= 0) & scope)
    log(f"[usidx] scope={ARGS.oop} · train 양성 {len(pos_idx):,} / 정상 {len(neg_idx):,} "
        f"({len(neg_idx)/max(len(pos_idx),1):.1f}:1)")

    Xn = EF[:n_tr][neg_idx]
    t0 = time.time()
    lab, dist, stats = fit_clusters(Xn, US_K, seed=42)
    log(f"[usidx] MiniBatchKMeans k={US_K} 적합 {time.time()-t0:.1f}s · "
        f"빈군집 {stats['empty_clusters']} · 크기 {stats['cluster_size_min']}~{stats['cluster_size_max']}")

    t0 = time.time()
    KREF = kld_reference(Xn)
    log(f"[usidx] KLD 모집단 기준 캐시 {time.time()-t0:.1f}s (후보마다 다시 안 잰다)")

    rows = []
    for ratio in (30.0, 100.0):
        n_target = int(round(len(pos_idx) * ratio))
        n_target = min(n_target, len(neg_idx))
        for si in range(US_N_SUBSETS):
            best = None
            for c in range(US_KLD_TRIES):
                sd = 42 + si * 100 + c
                sub = sample_indices(lab, dist, n_target, center_frac=0.0, seed=sd)
                k = _kld_dim(KREF, Xn[sub])
                if best is None or k < best[0]:
                    best = (k, sub, sd)
            kld, sub, sd = best
            keep = np.sort(np.concatenate([pos_idx, neg_idx[sub]]))
            nm = f"ensemble_cssmc_r{int(ratio)}_s{si}"
            np.save(CACHE / f"us_{nm}.npy", keep)
            rows.append(dict(name=nm, ratio=ratio, subset=si, seed=sd, kld_dim=kld,
                             n_keep=int(len(keep)), n_pos=int(len(pos_idx)),
                             n_neg=int(len(sub))))
            log(f"[usidx] {nm}: {len(keep):,}행 (정상 {len(sub):,}) KLD {kld:.3e} seed={sd}")

    # 대조군: 같은 크기 random
    rng = np.random.default_rng(42)
    for ratio in (30.0, 100.0):
        n_target = min(int(round(len(pos_idx) * ratio)), len(neg_idx))
        sub = np.sort(rng.choice(len(neg_idx), n_target, replace=False))
        keep = np.sort(np.concatenate([pos_idx, neg_idx[sub]]))
        nm = f"random_r{int(ratio)}_s0"
        np.save(CACHE / f"us_{nm}.npy", keep)
        rows.append(dict(name=nm, ratio=ratio, subset=0, seed=42,
                         kld_dim=_kld_dim(KREF, Xn[sub]), n_keep=int(len(keep)),
                         n_pos=int(len(pos_idx)), n_neg=int(len(sub))))
        log(f"[usidx] {nm}: {len(keep):,}행 KLD {rows[-1]['kld_dim']:.3e}")

    (CACHE / "us_index.json").write_text(
        json.dumps({"k": US_K, "scope": ARGS.oop, "cluster_stats": stats, "subsets": rows},
                   ensure_ascii=False, indent=2, default=float), encoding="utf-8")
    log(f"[usidx] 완료 → {CACHE}/us_index.json")

## §5. 그래프 구성 — 원본 build_window 과 수치적으로 동일

In [8]:
def load_us_keep(method, ratio, subset):
    """언더샘플 keep 인덱스 (train 좌표계 0..n_tr-1)."""
    p = CACHE / f"us_{method}_r{int(ratio)}_s{subset}.npy"
    if not p.exists():
        p2 = CACHE / f"us_{method}_r{int(ratio)}_s0.npy"
        if not p2.exists():
            raise FileNotFoundError(f"{p} 도 {p2} 도 없다. --stage usidx 를 먼저 돌려라.")
        p = p2
    return np.load(p)


# ══════════════════════════════════════════════════════════════════════════════
# §5. 그래프 구성 — 원본 build_window 과 **수치적으로 동일**하다.
#     바뀐 것은 배열을 전역에서 직접 읽던 것을 인자 A 로 받게 한 것뿐.
#     (--us prune 일 때 학습 구간만 압축된 배열을 먹이기 위해서다.)
#     us=none / us=mask 에서는 A 가 원본 전역 배열 그대로라 결과가 원본과 같다.
# ══════════════════════════════════════════════════════════════════════════════
class Arrays:
    def __init__(self, src, dst, ef, apaid, arecv, y):
        self.SRC, self.DST, self.EF = src, dst, ef
        self.A_PAID, self.A_RECV, self.Y = apaid, arecv, y


def window_ranges(lo, hi):
    return [(max(a - CONTEXT_EDGES, 0), a, min(a + WINDOW_SIZE, hi))
            for a in range(lo, hi, WINDOW_SIZE)]


def build_window(ctx_lo, lo, hi, A, keep_t=None, damage=None):
    m = hi - ctx_lo
    s, d = A.SRC[ctx_lo:hi], A.DST[ctx_lo:hi]
    nodes, inv = np.unique(np.concatenate([s, d]), return_inverse=True)
    inv = inv.reshape(-1)
    sl, dl = inv[:m].astype(np.int64), inv[m:].astype(np.int64)
    nn_ = len(nodes)
    tgt = slice(lo - ctx_lo, hi - ctx_lo)
    sl_t, dl_t = sl[tgt], dl[tgt]

    out_deg = np.bincount(sl, minlength=nn_).astype(np.float32)
    in_deg = np.bincount(dl, minlength=nn_).astype(np.float32)
    ap, ar = A.A_PAID[ctx_lo:hi], A.A_RECV[ctx_lo:hi]
    mean_out = (np.bincount(sl, weights=ap, minlength=nn_) / np.maximum(out_deg, 1)).astype(np.float32)
    mean_in = (np.bincount(dl, weights=ar, minlength=nn_) / np.maximum(in_deg, 1)).astype(np.float32)
    pair_f = sl * nn_ + dl

    cols = [np.log1p(out_deg), np.log1p(in_deg), mean_out, mean_in]
    if NODE_FEAT_DIM > 4:
        uf = np.unique(pair_f)
        n_out_nb = np.bincount(uf // nn_, minlength=nn_).astype(np.float32)
        n_in_nb = np.bincount(uf % nn_, minlength=nn_).astype(np.float32)
        deg = out_deg + in_deg
        cols += [np.log1p(n_out_nb), np.log1p(n_in_nb),
                 ((out_deg - in_deg) / np.maximum(deg, 1)).astype(np.float32),
                 (out_deg / np.maximum(n_out_nb, 1)).astype(np.float32),
                 (in_deg / np.maximum(n_in_nb, 1)).astype(np.float32)]
    x = np.stack(cols, axis=1).astype(np.float32)

    ei = np.stack([sl, dl])
    ei_full = np.concatenate([ei, ei[::-1]], axis=1)
    ea = A.EF[ctx_lo:hi]
    extra = np.zeros((2 * m, EDGE_EXTRA), dtype=np.float32)
    extra[:m, 0] = 1.0
    extra[m:, 1] = 1.0

    if USE_PORTS:
        o = np.argsort(pair_f, kind="stable")
        srt = pair_f[o]
        newg = np.r_[True, srt[1:] != srt[:-1]]
        gid = np.cumsum(newg) - 1
        start = np.flatnonzero(newg)
        rank = np.empty(m, np.int64)
        rank[o] = np.arange(m) - start[gid]
        mult_srt = np.bincount(gid, minlength=len(start)).astype(np.float32)[gid]
        mult = np.empty(m, np.float32)
        mult[o] = mult_srt
        extra[:m, 2] = (np.log1p(rank) / np.log(20)).astype(np.float32)
        extra[m:, 2] = extra[:m, 2]
        extra[:m, 3] = (np.log1p(mult) / np.log(20)).astype(np.float32)
        extra[m:, 3] = extra[:m, 3]

    ea_full = np.concatenate([np.concatenate([ea, ea], axis=0), extra], axis=1)
    pf_full = np.concatenate([pair_f, dl * nn_ + sl])
    pu, pinv = np.unique(pf_full, return_inverse=True)
    pair_dst = (pu % nn_).astype(np.int64)

    y_t = A.Y[lo:hi]
    node_y = ACCT_POS[nodes].astype(np.float32)
    node_mask = np.zeros(nn_, bool)
    if keep_t is None:
        node_mask[sl_t] = True
        node_mask[dl_t] = True
    else:
        # 손실 마스킹: 보조 노드 손실도 '남긴 타깃 엣지'가 닿는 노드로 제한한다.
        # 안 하면 노드 손실을 통해 버린 표본의 지도신호가 그대로 새어 들어온다.
        kk = keep_t.astype(bool)
        node_mask[sl_t[kk]] = True
        node_mask[dl_t[kk]] = True

    if damage is not None:
        deg_all = out_deg + in_deg
        # 1-hop 이웃: 타깃 엣지 양 끝 노드가 창 안에서 갖는 서로 다른 이웃 수
        uf2 = np.unique(pair_f)
        nb_out = np.bincount(uf2 // nn_, minlength=nn_)
        nb_in = np.bincount(uf2 % nn_, minlength=nn_)
        nb = (nb_out + nb_in).astype(np.float64)
        damage.append(dict(
            m=int(m), n_target=int(hi - lo), n_nodes=int(nn_),
            n_uniq_pairs=int(len(pu)), n_dir_edges=int(2 * m),
            mean_deg=float(deg_all.mean()),
            mean_nb_target=float(nb[np.concatenate([sl_t, dl_t])].mean()),
            iso_ratio_target=float((nb[np.concatenate([sl_t, dl_t])] <= 1).mean()),
        ))

    out = dict(x=torch.from_numpy(x), ei=torch.from_numpy(ei_full), ea=torch.from_numpy(ea_full),
               y=torch.from_numpy(y_t), m=m,
               pinv=torch.from_numpy(pinv.reshape(-1).astype(np.int64)),
               pair_dst=torch.from_numpy(pair_dst), n_pairs=len(pu),
               node_y=torch.from_numpy(node_y), node_mask=torch.from_numpy(node_mask), tgt=tgt)
    if keep_t is not None:
        out["keep"] = torch.from_numpy(keep_t.astype(np.float32))
    return out

## 원본 노트북 cell 7 — MegaConv·EdgeGINe 아키텍처 (한 글자도 안 바꿈)

In [9]:
# ═══ 원본 노트북 cell 7 (§5 아키텍처) — 한 글자도 안 바꿨다 ═══
def to_dev(g):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in g.items()}


def autocast_lowp(device_type):
    try:
        if torch.is_autocast_enabled(device_type):
            return torch.get_autocast_dtype(device_type)
    except TypeError:
        if device_type == "cuda" and torch.is_autocast_enabled():
            return torch.get_autocast_gpu_dtype()
    return torch.float32


class MegaConv(nn.Module):
    def __init__(self, hidden, mega=True, msg_dropout=0.0, agg_clamp=True, clamp_frac=0.25):
        super().__init__()
        self.mega = mega
        self.msg_dropout = float(msg_dropout)
        self.agg_clamp = bool(agg_clamp)
        self.clamp_frac = float(clamp_frac)
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.register_buffer("agg_absmax_t", torch.zeros(()), persistent=False)

    @property
    def agg_absmax(self):
        return float(self.agg_absmax_t.detach().cpu())

    def reset_diag(self):
        with torch.no_grad():
            self.agg_absmax_t.zero_()

    def forward(self, h, ei, e, pinv, pair_dst, n_pairs):
        msg = F.relu(h[ei[0]] + e)
        if self.msg_dropout > 0:
            msg = F.dropout(msg, self.msg_dropout, self.training)
        m32 = msg.float()

        if self.mega:
            pair = torch.zeros(n_pairs, m32.size(1), device=m32.device, dtype=torch.float32)
            pair = pair.index_add(0, pinv, m32)
            agg = torch.zeros(h.size(0), m32.size(1), device=m32.device, dtype=torch.float32)
            agg = agg.index_add(0, pair_dst, pair)
        else:
            agg = torch.zeros(h.size(0), m32.size(1), device=m32.device, dtype=torch.float32)
            agg = agg.index_add(0, ei[1], m32)

        if agg.numel():
            with torch.no_grad():
                self.agg_absmax_t.copy_(torch.maximum(self.agg_absmax_t, agg.detach().abs().amax()))

        lowp = autocast_lowp(agg.device.type)
        if self.agg_clamp and lowp != torch.float32:
            lim = torch.finfo(lowp).max * self.clamp_frac
            agg = agg.clamp(-lim, lim)

        return self.mlp((1 + self.eps) * h + agg.to(msg.dtype))


class EdgeGINe(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden=256, layers=2, dropout=0.15, mega=True,
                 msg_dropout=0.0, agg_clamp=True, clamp_frac=0.25):
        super().__init__()
        self.node_in = nn.Linear(node_dim, hidden)
        self.edge_in = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList(
            [MegaConv(hidden, mega, msg_dropout, agg_clamp, clamp_frac) for _ in range(layers)]
        )
        self.eupds = nn.ModuleList(
            [nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden)) for _ in range(layers)]
        )
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(3 * hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )
        self.node_head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 1))

    def reset_diag(self):
        for c in self.convs:
            c.reset_diag()

    def agg_headroom(self, dtype):
        vals = [c.agg_absmax for c in self.convs]
        return (max(vals) / torch.finfo(dtype).max if vals else 0.0), vals

    def forward(self, g):
        ei, m = g["ei"], g["m"]
        h = self.node_in(g["x"])
        e = self.edge_in(g["ea"])

        for conv, eupd, norm in zip(self.convs, self.eupds, self.norms):
            h = F.relu(norm(conv(h, ei, e, g["pinv"], g["pair_dst"], g["n_pairs"])))
            h = F.dropout(h, self.dropout, self.training)
            e = e + eupd(torch.cat([h[ei[0]], h[ei[1]], e], dim=1))

        tgt = g["tgt"]
        fi = ei[:, :m][:, tgt]
        z = torch.cat([h[fi[0]], h[fi[1]], e[:m][tgt]], dim=1)
        return self.head(z).squeeze(-1), self.node_head(h).squeeze(-1)


def make_model():
    return EdgeGINe(NODE_DIM, EDGE_DIM, HIDDEN, N_LAYERS, DROPOUT, USE_MEGA, MSG_DROPOUT, AGG_CLAMP, AGG_CLAMP_FRAC)

## 원본 노트북 cell 8 — 평가 프로토콜 (듀얼 예산 라우팅·패턴 재현율·report_row)

In [10]:
# ═══ 원본 노트북 cell 8 (§6 평가 프로토콜) — 한 글자도 안 바꿨다 ═══

# §6. 평가 프로토콜 및 [v13 핵심] 듀얼 예산 라우팅
def dual_budget_hits(score_main, score_b, seg, budget_total=ALERT_BUDGET, r=0.25):
    """
    [v13 Breakthrough] 예산 분리 라우팅
    (1-r) 예산은 score_main 상위 배정, 미선정 잔여 거래 중 r 예산은 score_b 상위 배정
    """
    n_seg = len(score_main)
    k_total = max(int(round(n_seg * budget_total)), 1)
    k_b = int(round(k_total * r))
    k_main = k_total - k_b

    # 1. Main 상위 선정
    thr_main = float(np.partition(score_main, n_seg - k_main)[n_seg - k_main]) if k_main > 0 else float("inf")
    hit_main = score_main >= thr_main

    if k_b <= 0:
        return hit_main

    # 2. Score B 상위 선정 (Main 미선정 거래 대상)
    hit_mask = hit_main.copy()
    unselected = np.flatnonzero(~hit_mask)
    if len(unselected) > 0:
        sub_scores = score_b[unselected]
        k_b_eff = min(k_b, len(sub_scores))
        thr_b = float(np.partition(sub_scores, len(sub_scores) - k_b_eff)[len(sub_scores) - k_b_eff])
        hit_b_sub = sub_scores >= thr_b
        hit_mask[unselected[hit_b_sub]] = True

    return hit_mask


def pattern_recall(prob, seg, budget=ALERT_BUDGET, hit=None):
    lo, hi = SEG_RANGE[seg]
    if hit is None:
        thr = float(np.quantile(prob, 1 - budget))
        hit = prob >= thr
    else:
        thr = float("nan")

    pos = np.flatnonzero(Y[lo:hi] > 0)
    in_p = PAT_MASK[lo:hi][pos]
    h = hit[pos]
    r_in = float(h[in_p].mean() * 100) if in_p.any() else float("nan")
    r_out = float(h[~in_p].mean() * 100) if (~in_p).any() else float("nan")

    cov = det = 0
    for b in PAT_BLOCKS:
        idxs = b["idxs"]
        sel = idxs[(idxs >= lo) & (idxs < hi)]
        if not len(sel):
            continue
        cov += 1
        if hit[sel - lo].any():
            det += 1

    return dict(thr=thr, R_in=r_in, R_out=r_out, n_in=int(in_p.sum()), n_out=int((~in_p).sum()),
                scen=(det / cov * 100 if cov else float("nan")), hit=hit)


def report_row(name, vp, tp, v_hit=None, t_hit=None):
    sv = pattern_recall(vp, "val", hit=v_hit)
    st = pattern_recall(tp, "test", hit=t_hit)
    yp = st["hit"].astype(int)
    return {
        "모델": name,
        "Val PR-AUC": float(average_precision_score(val_y, vp)),
        "Test PR-AUC": float(average_precision_score(test_y, tp)),
        "패턴내R@예산": st["R_in"], "패턴밖R@예산": st["R_out"],
        "시나리오R%": st["scen"],
        "P@예산": float(precision_score(test_y, yp, zero_division=0) * 100),
        "F1@예산": float(f1_score(test_y, yp, zero_division=0) * 100),
        "val패턴내R": sv["R_in"], "val패턴밖R": sv["R_out"],
    }


def rank01(a):
    return pd.Series(a).rank(pct=True, method="average").to_numpy()


def select_by_rule(tbl, floor, prauc_col="Val PR-AUC", oop_col="val패턴밖R", tag=""):
    t = tbl.copy()
    t["_oop"] = t[oop_col].fillna(-1.0)
    cand = t[t[prauc_col] >= floor]
    if not len(cand):
        cand = t

    hit = cand[cand["_oop"] >= TARGET_OUT_RECALL]
    if len(hit):
        best = hit.sort_values([prauc_col, "_oop"], ascending=False).iloc[0]
        how = f"목표 {TARGET_OUT_RECALL:g}% 달성 후보 {len(hit)}개 중 Val PR-AUC 최대"
    else:
        tied = cand[cand["_oop"] >= cand["_oop"].max() - OOP_TIE_TOL]
        best = tied.sort_values([prauc_col, "_oop"], ascending=False).iloc[0]
        how = f"목표 미달 → 후보 {len(cand)}개 중 val 패턴밖 최대치 ±{OOP_TIE_TOL:.2f}%p 동률군에서 Val PR-AUC 최대"

    print(f"[선택규칙{(' · ' + tag) if tag else ''}] Val PR-AUC ≥ {floor:.4f} · {how}")
    return best



# ══════════════════════════════════════════════════════════════════════════════

## §7. Track A GNN 학습

In [11]:
# §7. Track A GNN 학습
#   원본과의 유일한 차이: focal 손실에 keep 마스크를 곱할 수 있게 한 것.
#   keep=None 이면 (per).mean() 으로 원본과 완전히 동일한 식이 된다.
# ══════════════════════════════════════════════════════════════════════════════
def focal_bce_with_logits(logits, targets, pos_weight, gamma, keep=None):
    p = torch.sigmoid(logits)
    logp, log1mp = F.logsigmoid(logits), F.logsigmoid(-logits)
    loss_pos = -torch.pow(torch.clamp(1.0 - p, min=0.0), gamma) * logp
    loss_neg = -torch.pow(torch.clamp(p, min=0.0), gamma) * log1mp
    per = pos_weight * targets * loss_pos + (1.0 - targets) * loss_neg
    if keep is None:
        return per.mean()
    return (per * keep).sum() / keep.sum().clamp(min=1.0)


WCACHE_EVAL = {}


def get_eval_window(w, A):
    if w not in WCACHE_EVAL:
        WCACHE_EVAL[w] = build_window(*w, A)
    return WCACHE_EVAL[w]


def to_dev_g(g):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in g.items()}


def make_train_set(seed_pos):
    """시드별 학습 구간 그래프 + keep 마스크.

    반환: (Arrays, 창목록, keep_per_window[list or None], 통계dict, 훼손도 rows)
      us=none  : 원본 그대로 (scope=drop 이면 그 행만 손실에서 뺀다)
      us=mask  : 그래프 온전 · 손실에서만 제외  → 창 수 불변
      us=prune : 행 자체 삭제 · 창을 압축 배열 위에 다시 깖 → 창 수 감소
    """
    scope = train_scope_mask()
    damage = []
    if ARGS.us == "none":
        keep_np = scope.astype(np.float32) if ARGS.oop == "drop" else None
        A = Arrays(SRC, DST, EF, A_PAID, A_RECV, Y)
        W = window_ranges(0, n_tr)
        kw = None if keep_np is None else [keep_np[lo:hi] for (_, lo, hi) in W]
        stat = dict(n_train_rows=int(n_tr), n_windows=len(W),
                    n_loss_rows=int(n_tr if keep_np is None else keep_np.sum()))
    elif ARGS.us == "mask":
        sel = load_us_keep(ARGS.method, ARGS.ratio, seed_pos)
        sel = sel[scope[sel]]                   # scope=drop 이면 여기서도 빠진다
        keep_np = np.zeros(n_tr, dtype=np.float32)
        keep_np[sel] = 1.0
        A = Arrays(SRC, DST, EF, A_PAID, A_RECV, Y)
        W = window_ranges(0, n_tr)
        kw = [keep_np[lo:hi] for (_, lo, hi) in W]
        stat = dict(n_train_rows=int(n_tr), n_windows=len(W), n_loss_rows=int(len(sel)))
    else:
        # prune — 엣지(행) 실제 삭제
        sel = load_us_keep(ARGS.method, ARGS.ratio, seed_pos)
        sel = sel[scope[sel]]
        A = Arrays(SRC[sel], DST[sel], EF[sel], A_PAID[sel], A_RECV[sel], Y[sel])
        W = window_ranges(0, len(sel))
        kw = None
        stat = dict(n_train_rows=int(len(sel)), n_windows=len(W), n_loss_rows=int(len(sel)))

    # 창을 **한 번만** 만든다 (훼손도 측정과 학습이 같은 창을 쓴다)
    WT = [build_window(c, lo, hi, A, None if kw is None else kw[i], damage)
          for i, (c, lo, hi) in enumerate(W)]
    return A, W, kw, stat, damage, WT


def train_one_seed(seed, seed_pos):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    t_w = time.time()
    A_tr, TRAIN_W, KW, stat, damage, WT = make_train_set(seed_pos)
    # 창은 원본 노트북과 똑같이 CPU 에 두고 스텝마다 올린다.
    # (31창을 미리 GPU 에 올리면 정적으로만 3.9GB 를 물어 OOM 위험이 생긴다.)
    t_win = time.time() - t_w
    log(f"  seed {seed}: 학습창 {len(WT)}개 · 학습행 {stat['n_train_rows']:,} · "
        f"손실행 {stat['n_loss_rows']:,} · 창빌드 {t_win:.1f}s")

    ytr_eff = A_tr.Y[:stat['n_train_rows']] if ARGS.us == "prune" else Y[:n_tr]
    if KW is None:
        pos = float((ytr_eff > 0).sum())
        tot = float(len(ytr_eff))
    else:
        km = np.concatenate(KW) > 0
        pos = float((Y[:n_tr][km] > 0).sum())
        tot = float(km.sum())
    pw_raw = (tot - pos) / max(pos, 1.0)
    model = make_model().to(DEVICE)
    pw = torch.tensor(min(pw_raw, POS_WEIGHT_CAP), device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5,
                                                       patience=3, min_lr=1e-5)
    amp_on = USE_AMP and DEVICE.type == "cuda"
    amp_dt = AMP_DTYPE if amp_on else torch.float32
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=(amp_on and amp_dt == torch.float16))
    A_full = Arrays(SRC, DST, EF, A_PAID, A_RECV, Y)

    @torch.no_grad()
    def predict(windows):
        model.eval()
        ps = []
        for w in windows:
            g = to_dev_g(get_eval_window(w, A_full))
            with torch.autocast(DEVICE.type, amp_dt, enabled=amp_on):
                logit, _ = model(g)
            ps.append(torch.sigmoid(logit.float()).cpu().numpy())
        return np.concatenate(ps)

    best_ap, best_state, best_ep, left = -1.0, None, 0, PATIENCE
    ep_times, hist = [], []
    for epoch in range(1, EPOCHS + 1):
        te = time.time()
        model.train()
        model.reset_diag()
        ep_loss = 0.0
        for wi in np.random.permutation(len(WT)):
            g = to_dev_g(WT[wi])
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, amp_dt, enabled=amp_on):
                el, nl = model(g)
                loss = focal_bce_with_logits(el.float(), g["y"], pw, FOCAL_GAMMA,
                                             g.get("keep"))
                if AUX_NODE_W > 0 and USE_NODE_AUX:
                    nm = g["node_mask"]
                    if bool(nm.any()):
                        ny = g["node_y"][nm]
                        pwn = torch.clamp(
                            (nm.sum().float() - ny.sum()) / torch.clamp(ny.sum(), min=1.0),
                            max=POS_WEIGHT_CAP)
                        loss += AUX_NODE_W * F.binary_cross_entropy_with_logits(
                            nl[nm].float(), ny, pos_weight=pwn)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            ep_loss += float(loss.detach())
        t_fit = time.time() - te
        val_ap = float(average_precision_score(val_y, predict(VAL_W)))
        sched.step(val_ap)
        ep_times.append(t_fit)
        hist.append(dict(epoch=epoch, loss=ep_loss / max(len(WT), 1), val_ap=val_ap,
                         fit_s=round(t_fit, 2), total_s=round(time.time() - te, 2)))
        log(f"    ep{epoch:02d} loss {ep_loss/max(len(WT),1):.4f} valAP {val_ap:.4f} "
            f"({t_fit:.1f}s 학습 / {time.time()-te:.1f}s 총)")
        if val_ap > best_ap:
            best_ap, best_ep, left = val_ap, epoch, PATIENCE
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            left -= 1
            if left == 0:
                log(f"    early stop (best ep{best_ep} AP {best_ap:.4f})")
                break

    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{ARGS.tag}_trackA_seed{seed}.pt")
    res = dict(seed=seed, val_ap=float(best_ap), best_epoch=best_ep,
               epochs_run=len(ep_times), sec_per_epoch=float(np.mean(ep_times)),
               fit_seconds=float(np.sum(ep_times)), window_build_s=round(t_win, 1),
               pos_weight_raw=float(pw_raw), pos_weight_used=float(min(pw_raw, POS_WEIGHT_CAP)),
               history=hist, **stat)
    vp, tp = predict(VAL_W), predict(TEST_W)
    WT.clear()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return res, vp, tp, damage


def stage_trackA():
    global ACCT_POS, VAL_W, TEST_W
    ACCT_POS = np.zeros(NUM_ACCOUNTS, dtype=bool)
    _tp = Y[:n_tr] > 0
    ACCT_POS[SRC[:n_tr][_tp]] = True
    ACCT_POS[DST[:n_tr][_tp]] = True
    VAL_W, TEST_W = window_ranges(n_tr, n_va), window_ranges(n_va, n)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log(f"===== Track A · tag={ARGS.tag} us={ARGS.us} oop={ARGS.oop} "
        f"method={ARGS.method} ratio={ARGS.ratio:g} seeds={SEED_LIST} epochs={EPOCHS} =====")
    t0 = time.time()
    runs, vps, tps, dmg = [], [], [], None
    for i, s in enumerate(SEED_LIST):
        r, vp, tp, d = train_one_seed(s, i)
        runs.append(r)
        vps.append(vp)
        tps.append(tp)
        if dmg is None:
            dmg = d
    GNN_V = np.mean(vps, axis=0)
    GNN_T = np.mean(tps, axis=0)
    np.save(OUT / f"{ARGS.tag}_gnn_val.npy", GNN_V)
    np.save(OUT / f"{ARGS.tag}_gnn_test.npy", GNN_T)
    row = report_row(f"Track A · GNN 앙상블({len(runs)}시드)", GNN_V, GNN_T)

    dm = pd.DataFrame(dmg)
    dm.to_csv(OUT / f"damage_{ARGS.tag}.csv", index=False)
    dmg_summary = dict(
        n_windows=int(len(dm)), mean_deg=float(dm["mean_deg"].mean()),
        mean_nb_target=float(dm["mean_nb_target"].mean()),
        iso_ratio_target=float(dm["iso_ratio_target"].mean()),
        total_nodes=int(dm["n_nodes"].sum()), total_uniq_pairs=int(dm["n_uniq_pairs"].sum()),
        total_dir_edges=int(dm["n_dir_edges"].sum()))

    out = dict(tag=ARGS.tag, us=ARGS.us, oop=ARGS.oop, method=ARGS.method,
               ratio=ARGS.ratio, seeds=SEED_LIST, epochs_cap=EPOCHS, patience=PATIENCE,
               wall_seconds=round(time.time() - t0, 1),
               gpu_peak_alloc_GB=(round(torch.cuda.max_memory_allocated() / 1e9, 2)
                                  if DEVICE.type == "cuda" else None),
               gpu_peak_reserved_GB=(round(torch.cuda.max_memory_reserved() / 1e9, 2)
                                     if DEVICE.type == "cuda" else None),
               runs=runs, row_trackA=row, graph_damage=dmg_summary)
    (OUT / f"{ARGS.tag}_trackA.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
    print("\n" + pd.DataFrame([row]).round(4).to_string(index=False))
    log(f"Track A 완료 {out['wall_seconds']:.1f}s → {OUT}/{ARGS.tag}_trackA.json")
    return out


# ══════════════════════════════════════════════════════════════════════════════

## §8. Track B — 원본은 좌표하강 15칸 스윕. 기본은 1칸 고정(`trackb_sweep=True`로 복원)

In [12]:
# §8. Track B — 원본은 좌표하강 15칸 스윕. 여기서는 **1칸 고정**이 기본이다.
#     근거: 3.05M행 × 106차원 × 1500트리 1적합이 수 분이라 15칸이면 마감 안에 못 돈다.
#     원본 코드의 sw 는 n_tr 을 하드코딩해 언더샘플을 넣으면 길이 불일치로 터진다.
#     여기서는 실제 선택된 행 수로 만든다(원본 버그 수정).
# ══════════════════════════════════════════════════════════════════════════════
LGB_BASE_CFG = dict(w=1.0, lr=0.03, leaves=127, mcs=20, oop=1.0)


def stage_trackB():
    XTAB = np.load(CACHE / "XTAB.npy")
    SRC_SEQ = np.load(CACHE / "SRC_SEQ.npy")
    DST_SEQ = np.load(CACHE / "DST_SEQ.npy")
    Y_INT = Y.astype(np.int32)

    scope = train_scope_mask()
    if ARGS.us == "none":
        rows_tr = np.flatnonzero(scope)
    else:
        sel = load_us_keep(ARGS.method, ARGS.ratio, 0)
        rows_tr = sel[scope[sel]]
    XB_TR, YB_TR = XTAB[rows_tr], Y_INT[rows_tr]
    XB_VA, YB_VA = XTAB[n_tr:n_va], Y_INT[n_tr:n_va]
    XB_TE = XTAB[n_va:]
    IS_ISO = (SRC_SEQ[rows_tr] <= 0.05) & (DST_SEQ[rows_tr] <= 0.05)
    PAT_TR = PAT_MASK[rows_tr]
    log(f"===== Track B · 행 {len(rows_tr):,} (양성 {int((YB_TR>0).sum()):,}) · "
        f"{XB_TR.shape[1]}차원 · 1칸 고정 {LGB_BASE_CFG} =====")

    def fit(w, lr, leaves, mcs, oop=1.0, y=None, seed=42):
        yy = YB_TR if y is None else y
        sw = np.ones(len(rows_tr), dtype=np.float32)
        sw = np.where(IS_ISO, sw * 2.0, sw)
        if oop != 1.0:
            sw = np.where((yy > 0) & (~PAT_TR), sw * float(oop), sw)
        mdl = lgb.LGBMClassifier(
            objective="binary", metric="average_precision",
            n_estimators=LGB_N_ESTIMATORS, learning_rate=lr, num_leaves=leaves,
            min_child_samples=mcs, scale_pos_weight=float(w), subsample=0.9,
            subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, max_bin=255,
            n_jobs=-1, random_state=seed, verbose=-1)
        return mdl, sw, yy

    t0 = time.time()
    mdl, sw, yy = fit(**LGB_BASE_CFG)
    ev_y = YB_VA
    mdl.fit(XB_TR, yy, sample_weight=sw, eval_set=[(XB_VA, ev_y)],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(LGB_EARLY_STOP, first_metric_only=True, verbose=False),
                       lgb.log_evaluation(0)])
    TAB_V = mdl.predict_proba(XB_VA)[:, 1]
    TAB_T = mdl.predict_proba(XB_TE)[:, 1]
    t_main = time.time() - t0
    np.save(OUT / f"{ARGS.tag}_tab_val.npy", TAB_V)
    np.save(OUT / f"{ARGS.tag}_tab_test.npy", TAB_T)
    pr = pattern_recall(TAB_V, "val")
    row = report_row("Track B · LightGBM (RuleBoost OFF · 1칸 고정)", TAB_V, TAB_T)
    log(f"Track B main: valAP {row['Val PR-AUC']:.4f} · 패턴내R {pr['R_in']:.2f}% / "
        f"패턴밖R {pr['R_out']:.2f}% · best_iter {mdl.best_iteration_} · {t_main:.1f}s")

    carve = None
    if ARGS.oop == "keep":
        Y_OOP_TR = ((Y[rows_tr] > 0) & (~PAT_TR)).astype(np.int32)
        Y_OOP_VA = ((Y[n_tr:n_va] > 0) & (~PAT_MASK[n_tr:n_va])).astype(np.int32)
        if Y_OOP_TR.sum() >= 20 and Y_OOP_VA.sum() >= 5:
            t1 = time.time()
            m2, sw2, _ = fit(**LGB_BASE_CFG, y=Y_OOP_TR)
            m2.fit(XB_TR, Y_OOP_TR, eval_set=[(XB_VA, Y_OOP_VA)],
                   eval_metric="average_precision",
                   callbacks=[lgb.early_stopping(LGB_EARLY_STOP, first_metric_only=True,
                                                 verbose=False), lgb.log_evaluation(0)])
            OOP_V = m2.predict_proba(XB_VA)[:, 1]
            OOP_T = m2.predict_proba(XB_TE)[:, 1]
            np.save(OUT / f"{ARGS.tag}_carve_val.npy", OOP_V)
            np.save(OUT / f"{ARGS.tag}_carve_test.npy", OOP_T)
            opr = pattern_recall(OOP_V, "val")
            carve = dict(n_pos_train=int(Y_OOP_TR.sum()),
                         ap=float(average_precision_score(Y_OOP_VA, OOP_V)),
                         solo_R_out=opr["R_out"], solo_R_in=opr["R_in"],
                         fit_s=round(time.time() - t1, 1))
            log(f"패턴밖 전용 헤드: 양성 {carve['n_pos_train']:,} · AP {carve['ap']:.4f} · "
                f"단독 예산 패턴밖R {carve['solo_R_out']:.2f}% · {carve['fit_s']}s")

    out = dict(tag=ARGS.tag, cfg=LGB_BASE_CFG, sweep=False, us=ARGS.us, oop=ARGS.oop,
               n_train_rows=int(len(rows_tr)), n_train_pos=int((YB_TR > 0).sum()),
               best_iteration=int(mdl.best_iteration_ or LGB_N_ESTIMATORS),
               fit_seconds=round(t_main, 1), row_trackB=row,
               val_R_in=pr["R_in"], val_R_out=pr["R_out"], carve=carve)
    (OUT / f"{ARGS.tag}_trackB.json").write_text(
        json.dumps(out, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
    return out


# ══════════════════════════════════════════════════════════════════════════════

## §9~§10. 하이브리드 결합 + 듀얼 예산 — 원본 로직 그대로, 저장된 확률만 읽음

In [13]:
# §9~§10. 하이브리드 결합 + 듀얼 예산 — 원본 로직 그대로. 저장된 확률만 읽는다.
# ══════════════════════════════════════════════════════════════════════════════
def combine_scores(mode, w, ra, rb):
    if mode == "lin":
        return (1.0 - w) * ra + w * rb
    if mode == "max":
        return np.maximum((1.0 - w) * ra, w * rb)
    if mode == "maxq":
        if w <= 0:
            return ra.copy()
        return np.maximum(ra, np.clip((rb - (1.0 - w)) / w, 0.0, 1.0))
    raise ValueError(mode)


def select_v14(tbl, floor, in_floor=88.0, tag=""):
    t = tbl.copy()
    t["_oop"] = t["val패턴밖R"].fillna(-1.0)
    t["_in"] = t["val패턴내R"].fillna(-1.0)
    cand = t[t["Val PR-AUC"] >= floor]
    fnote = f"PR-AUC ≥ {floor:.4f} → {len(cand)}개"
    if not len(cand):
        cand = t
        fnote = f"PR-AUC ≥ {floor:.4f} 충족 0개 → 하한 해제"
    g = cand[cand["_in"] >= in_floor]
    if len(g):
        gnote = f"패턴내R ≥ {in_floor:.1f}% → {len(g)}개"
    else:
        eff = float(cand["_in"].max())
        g = cand[cand["_in"] >= eff - 1e-9]
        gnote = f"⚠ 패턴내R 충족 0개 → 최댓값 {eff:.2f}% 로 완화 → {len(g)}개"
    hit = g[g["_oop"] >= TARGET_OUT_RECALL_V14]
    if len(hit):
        best = hit.sort_values(["Val PR-AUC", "_oop"], ascending=False).iloc[0]
        how = f"패턴밖 목표 {TARGET_OUT_RECALL_V14:g}% 달성 {len(hit)}개 중 PR-AUC 최대"
    else:
        tied = g[g["_oop"] >= g["_oop"].max() - OOP_TIE_TOL]
        best = tied.sort_values(["Val PR-AUC", "_oop"], ascending=False).iloc[0]
        how = (f"패턴밖 목표 미달 → 최댓값 {g['_oop'].max():.2f}% "
               f"±{OOP_TIE_TOL:.2f}%p 동률군 {len(tied)}개 중 PR-AUC 최대")
    print(f"[선택 · {tag}] {fnote} | {gnote} | {how}")
    return best


TARGET_OUT_RECALL_V14 = 12.0


def stage_hybrid():
    ta = ARGS.hybrid_a or ARGS.tag
    tb = ARGS.hybrid_b
    GNN_V = np.load(OUT / f"{ta}_gnn_val.npy")
    GNN_T = np.load(OUT / f"{ta}_gnn_test.npy")
    TAB_V = np.load(OUT / f"{tb}_tab_val.npy")
    TAB_T = np.load(OUT / f"{tb}_tab_test.npy")
    ROW_A = report_row(f"Track A · GNN 앙상블 [{ta}]", GNN_V, GNN_T)
    ROW_B = report_row(f"Track B · LightGBM 1칸 [{tb}]", TAB_V, TAB_T)

    RG_V, RG_T = rank01(GNN_V), rank01(GNN_T)
    RB_V, RB_T = rank01(TAB_V), rank01(TAB_T)
    PRAUC_FLOOR = (float(TARGET_VAL_PRAUC) if ROW_A["Val PR-AUC"] >= TARGET_VAL_PRAUC
                   else float(ROW_A["Val PR-AUC"]) - HYBRID_PRAUC_TOL)
    print(f"[결합 하한] Track A Val PR-AUC={ROW_A['Val PR-AUC']:.4f} → floor={PRAUC_FLOOR:.4f}")

    HY = []
    for mode in ("lin", "max", "maxq"):
        for w in HYBRID_W_GRID:
            cv = combine_scores(mode, w, RG_V, RB_V)
            ct = combine_scores(mode, w, RG_T, RB_T)
            pr = pattern_recall(cv, "val")
            HY.append({"결합": mode, "w1": 1.0 - w, "w2": w,
                       "Val PR-AUC": float(average_precision_score(val_y, cv)),
                       "Test PR-AUC": float(average_precision_score(test_y, ct)),
                       "val패턴내R": pr["R_in"], "val패턴밖R": pr["R_out"]})
    TBL_HY = pd.DataFrame(HY)
    TBL_HY.to_csv(OUT / f"hybrid_grid_{ta}.csv", index=False)
    BEST = select_v14(TBL_HY, PRAUC_FLOOR, tag="하이브리드 랭킹")
    MAIN_V = combine_scores(BEST["결합"], BEST["w2"], RG_V, RB_V)
    MAIN_T = combine_scores(BEST["결합"], BEST["w2"], RG_T, RB_T)
    ROW_H = report_row(f"하이브리드 결합 ({BEST['결합']}, w2={BEST['w2']:.2f})", MAIN_V, MAIN_T)

    # w2=0.00 절제값 = Track A 단독. Track B 가 패턴내 성능에 기여하는지 여기서 바로 보인다.
    ABL = TBL_HY[(TBL_HY["결합"] == "lin") & (TBL_HY["w2"].isin([0.0, 0.25, 0.30, 0.50, 1.0]))]

    cp = OUT / f"{tb}_carve_val.npy"
    if cp.exists():
        CARVE_V = rank01(np.load(cp))
        CARVE_T = rank01(np.load(OUT / f"{tb}_carve_test.npy"))
        CTAG = "패턴밖 전용 헤드"
    else:
        CARVE_V, CARVE_T, CTAG = RB_V, RB_T, "Track B 전체"
    print(f"[듀얼 예산] carve-out 점수원 = {CTAG}")

    RGRID = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
    RR = []
    for r in RGRID:
        vh = dual_budget_hits(MAIN_V, CARVE_V, "val", ALERT_BUDGET, r)
        th = dual_budget_hits(MAIN_T, CARVE_T, "test", ALERT_BUDGET, r)
        rv, rt = pattern_recall(MAIN_V, "val", hit=vh), pattern_recall(MAIN_T, "test", hit=th)
        RR.append({"r": r, "Val PR-AUC": ROW_H["Val PR-AUC"], "Test PR-AUC": ROW_H["Test PR-AUC"],
                   "val패턴내R": rv["R_in"], "val패턴밖R": rv["R_out"],
                   "패턴내R@예산": rt["R_in"], "패턴밖R@예산": rt["R_out"], "시나리오R%": rt["scen"],
                   "P@예산": float(precision_score(test_y, th.astype(int), zero_division=0) * 100),
                   "F1@예산": float(f1_score(test_y, th.astype(int), zero_division=0) * 100),
                   "_vh": vh, "_th": th})
    TBL_R = pd.DataFrame(RR)
    print("\n" + TBL_R.drop(columns=["_vh", "_th"]).round(4).to_string(index=False))
    BR = select_v14(TBL_R, PRAUC_FLOOR, tag="듀얼 예산 라우팅")
    ROW_V = report_row(f"v14 (Dual Budget r={BR['r']:.2f}, carve={CTAG})",
                       MAIN_V, MAIN_T, v_hit=BR["_vh"], t_hit=BR["_th"])

    FINAL = pd.DataFrame([ROW_A, ROW_B, ROW_H, ROW_V])[
        ["모델", "Val PR-AUC", "Test PR-AUC", "패턴내R@예산", "패턴밖R@예산",
         "시나리오R%", "P@예산", "F1@예산"]]
    print("\n" + "=" * 95)
    print("── 표4-1. 최종 성능 비교 (동일 분할 · 상위 0.1% 예산) ──")
    print("=" * 95)
    print(FINAL.round(4).to_string(index=False))
    print("\n── w2 절제 (lin) : Track B 기여도 ──")
    print(ABL.round(4).to_string(index=False))

    NB = {"Val PR-AUC": 0.6606, "Test PR-AUC": 0.5997, "패턴내R@예산": 88.5965,
          "패턴밖R@예산": 4.7930, "F1@예산": 58.2020}
    print("\n── 노트북 (2) 실측치 대비 ──")
    for k, v in NB.items():
        print(f"  {k:14s} 노트북 {v:10.4f} | 재실행 {float(ROW_V[k]):10.4f} | Δ {float(ROW_V[k])-v:+10.4f}")

    res = dict(track_a_tag=ta, track_b_tag=tb, best_hybrid=BEST.drop(labels=[]).to_dict(),
               best_r=float(BR["r"]), carve_source=CTAG,
               rows={"trackA": ROW_A, "trackB": ROW_B, "hybrid": ROW_H, "v14": ROW_V},
               ablation_w2=ABL.to_dict("records"),
               routing=TBL_R.drop(columns=["_vh", "_th"]).to_dict("records"),
               notebook_reference=NB,
               delta_vs_notebook={k: float(ROW_V[k]) - v for k, v in NB.items()})
    (OUT / f"hybrid_{ta}.json").write_text(
        json.dumps(res, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
    log(f"하이브리드 완료 → {OUT}/hybrid_{ta}.json")
    return res


# ══════════════════════════════════════════════════════════════════════════════

## 실행

터미널 `--stage` 인자 대신, 아래 셀을 필요한 것만 순서대로 실행한다.

1. **최초 1회**: `build_cache()` — 원본 데이터 로드·정제·피처 생성 (실측 약 250초, 캐시에 저장됨)
2. 매 세션 시작 시: `load_cache()` — 저장된 캐시를 메모리로 로드 (수 초)
3. 새 언더샘플 방식/비율을 쓸 때만: `build_us_indices()`
4. `CONFIG`의 `us`/`ratio`/`oop`/`tag`를 원하는 값으로 바꾼 뒤 `stage_trackA()` — GPU, 1회 약 15~20분
5. `stage_trackB()` — CPU, 1회 약 14분
6. `CONFIG`의 `hybrid_a`/`hybrid_b`를 채운 뒤 `stage_hybrid()`

> GPU는 1장이므로 `stage_trackA()`를 여러 커널에서 동시에 돌리지 않는다.

In [14]:
build_cache()

중복 제거 9건 / 꼬리 절단 1,108건 / 최종 5,077,228건 (세탁률 0.0891%)
분할: train 3,046,336 / val 1,015,446 / test 1,015,446 | 고유 계좌 515,078개 [19.8s]
공통 엣지 피처: 64차원 완성
[Track B v14] 신규 피처 포함 총 106차원 (공통 64 + 행동이상치 42) [35.0s]
[v14 신규 18종] 유한성 OK | 오염도 점등(val) 11.16% | 역방향 점등(val) 2.26%
패턴 매칭 완료: Val 양성 1082건 / Test 양성 1143건 (동률허용폭 0.26%p)
[   131.4s] 캐시 덤프 시작
[   145.9s] 캐시 저장 완료 → /workspace/us_v14/cache


In [15]:
load_cache()

[   146.4s] 캐시 로드: n=5,077,228 train=3,046,336 val=1,015,446 test=1,015,446 | EF (5077228, 64) | 블록 363


In [16]:
# 새 언더샘플 방식/비율을 처음 쓸 때만 실행
build_us_indices()

[   146.5s] [usidx] scope=keep · train 양성 2,297 / 정상 3,044,039 (1325.2:1)
[   154.2s] [usidx] MiniBatchKMeans k=32 적합 7.3s · 빈군집 0 · 크기 19233~251172
[   159.3s] [usidx] KLD 모집단 기준 캐시 5.2s (후보마다 다시 안 잰다)
[   164.0s] [usidx] ensemble_cssmc_r30_s0: 71,207행 (정상 68,910) KLD 1.545e-05 seed=45
[   168.5s] [usidx] ensemble_cssmc_r30_s1: 71,207행 (정상 68,910) KLD 1.636e-05 seed=144
[   173.1s] [usidx] ensemble_cssmc_r30_s2: 71,207행 (정상 68,910) KLD 2.024e-05 seed=243
[   178.0s] [usidx] ensemble_cssmc_r100_s0: 231,997행 (정상 229,700) KLD 5.478e-06 seed=42
[   182.9s] [usidx] ensemble_cssmc_r100_s1: 231,997행 (정상 229,700) KLD 6.087e-06 seed=144
[   187.9s] [usidx] ensemble_cssmc_r100_s2: 231,997행 (정상 229,700) KLD 4.813e-06 seed=243
[   188.0s] [usidx] random_r30_s0: 71,207행 KLD 2.326e-05
[   188.1s] [usidx] random_r100_s0: 231,997행 KLD 5.277e-06
[   188.1s] [usidx] 완료 → /workspace/us_v14/cache/us_index.json


In [17]:
# CONFIG['us']/CONFIG['ratio']/CONFIG['oop']/CONFIG['tag'] 를 먼저 바꾼 뒤 실행
stage_trackA()

[   188.1s] ===== Track A · tag=base us=none oop=keep method=ensemble_cssmc ratio=30 seeds=[0, 1] epochs=25 =====
[   196.3s]   seed 0: 학습창 31개 · 학습행 3,046,336 · 손실행 3,046,336 · 창빌드 8.2s
[   219.2s]     ep01 loss 0.1793 valAP 0.0139 (17.7s 학습 / 22.0s 총)
[   238.3s]     ep02 loss 0.1283 valAP 0.0546 (17.7s 학습 / 19.0s 총)
[   257.6s]     ep03 loss 0.1236 valAP 0.2927 (17.6s 학습 / 18.9s 총)
[   276.4s]     ep04 loss 0.1189 valAP 0.3980 (17.4s 학습 / 18.6s 총)
[   295.1s]     ep05 loss 0.1150 valAP 0.5103 (17.4s 학습 / 18.7s 총)
[   314.1s]     ep06 loss 0.1122 valAP 0.5121 (17.6s 학습 / 18.9s 총)
[   332.7s]     ep07 loss 0.1112 valAP 0.5361 (17.3s 학습 / 18.5s 총)
[   352.0s]     ep08 loss 0.1107 valAP 0.5454 (18.0s 학습 / 19.4s 총)
[   370.6s]     ep09 loss 0.1101 valAP 0.5573 (17.3s 학습 / 18.5s 총)
[   389.4s]     ep10 loss 0.1081 valAP 0.5769 (17.2s 학습 / 18.5s 총)
[   408.4s]     ep11 loss 0.1078 valAP 0.5881 (17.5s 학습 / 18.7s 총)
[   427.5s]     ep12 loss 0.1071 valAP 0.5899 (17.5s 학습 / 18.8s 총)
[   446.4

{'tag': 'base',
 'us': 'none',
 'oop': 'keep',
 'method': 'ensemble_cssmc',
 'ratio': 30.0,
 'seeds': [0, 1],
 'epochs_cap': 25,
 'patience': 5,
 'wall_seconds': 967.4,
 'gpu_peak_alloc_GB': 7.05,
 'gpu_peak_reserved_GB': 33.59,
 'runs': [{'seed': 0,
   'val_ap': 0.6341313765639658,
   'best_epoch': 24,
   'epochs_run': 25,
   'sec_per_epoch': 17.436088104248046,
   'fit_seconds': 435.9022026062012,
   'window_build_s': 8.2,
   'pos_weight_raw': 1325.2237701349586,
   'pos_weight_used': 5.0,
   'history': [{'epoch': 1,
     'loss': 0.17930769992451515,
     'val_ap': 0.013902675850099227,
     'fit_s': 17.73,
     'total_s': 21.98},
    {'epoch': 2,
     'loss': 0.1283426253545669,
     'val_ap': 0.05462800642298701,
     'fit_s': 17.67,
     'total_s': 18.99},
    {'epoch': 3,
     'loss': 0.12357656249115544,
     'val_ap': 0.2927114245234139,
     'fit_s': 17.64,
     'total_s': 18.95},
    {'epoch': 4,
     'loss': 0.11887725130204231,
     'val_ap': 0.39803781681140793,
     'fit_

In [18]:
stage_trackB()

[  1157.5s] ===== Track B · 행 3,046,336 (양성 2,297) · 106차원 · 1칸 고정 {'w': 1.0, 'lr': 0.03, 'leaves': 127, 'mcs': 20, 'oop': 1.0} =====


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[  1790.9s] Track B main: valAP 0.6273 · 패턴내R 88.60% / 패턴밖R 4.11% · best_iter 994 · 632.7s


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[  1887.8s] 패턴밖 전용 헤드: 양성 1,120 · AP 0.0218 · 단독 예산 패턴밖R 14.91% · 96.9s


{'tag': 'base',
 'cfg': {'w': 1.0, 'lr': 0.03, 'leaves': 127, 'mcs': 20, 'oop': 1.0},
 'sweep': False,
 'us': 'none',
 'oop': 'keep',
 'n_train_rows': 3046336,
 'n_train_pos': 2297,
 'best_iteration': 994,
 'fit_seconds': 632.7,
 'row_trackB': {'모델': 'Track B · LightGBM (RuleBoost OFF · 1칸 고정)',
  'Val PR-AUC': 0.6273199857673158,
  'Test PR-AUC': 0.5550014955577198,
  '패턴내R@예산': 81.72514619883042,
  '패턴밖R@예산': 5.88235294117647,
  '시나리오R%': 95.08196721311475,
  'P@예산': 57.677165354330704,
  'F1@예산': 54.28439092172302,
  'val패턴내R': 88.60028860028861,
  'val패턴밖R': 4.113110539845758},
 'val_R_in': 88.60028860028861,
 'val_R_out': 4.113110539845758,
 'carve': {'n_pos_train': 1120,
  'ap': 0.02180605672738356,
  'solo_R_out': 14.910025706940875,
  'solo_R_in': 1.1544011544011543,
  'fit_s': 96.9}}

In [19]:
# CONFIG['hybrid_a']/CONFIG['hybrid_b'] 를 먼저 채운 뒤 실행
stage_hybrid()

[결합 하한] Track A Val PR-AUC=0.6365 → floor=0.6265
[선택 · 하이브리드 랭킹] PR-AUC ≥ 0.6265 → 63개 | 패턴내R ≥ 88.0% → 63개 | 패턴밖 목표 미달 → 최댓값 5.14% ±0.26%p 동률군 7개 중 PR-AUC 최대
[듀얼 예산] carve-out 점수원 = 패턴밖 전용 헤드

   r  Val PR-AUC  Test PR-AUC  val패턴내R  val패턴밖R  패턴내R@예산  패턴밖R@예산  시나리오R%    P@예산   F1@예산
0.00      0.6596       0.5982  92.9293   5.1414  89.6199   4.1394 96.1749 62.2660 58.5728
0.02      0.6596       0.5982  92.9293   4.6272  89.4737   4.1394 96.1749 62.1675 58.4801
0.04      0.6596       0.5982  92.7850   4.3702  89.1813   4.3573 96.1749 62.0690 58.3874
0.06      0.6596       0.5982  92.6407   4.3702  89.0351   4.1394 96.1749 61.8719 58.2020
0.08      0.6596       0.5982  92.6407   4.6272  88.3041   4.3573 96.1749 61.4778 57.8313
0.10      0.6596       0.5982  92.3521   4.6272  87.8655   4.7930 96.1749 61.3793 57.7386
0.12      0.6596       0.5982  92.0635   4.8843  87.7193   5.4466 96.1749 61.5764 57.9240
0.15      0.6596       0.5982  91.6306   4.3702  87.2807   6.3181 96.1749 61.6749 58.0

{'track_a_tag': 'base',
 'track_b_tag': 'base',
 'best_hybrid': {'결합': 'lin',
  'w1': 0.6,
  'w2': 0.4,
  'Val PR-AUC': 0.659639616493877,
  'Test PR-AUC': 0.598213327465676,
  'val패턴내R': 92.92929292929293,
  'val패턴밖R': 5.141388174807198,
  '_oop': 5.141388174807198,
  '_in': 92.92929292929293},
 'best_r': 0.0,
 'carve_source': '패턴밖 전용 헤드',
 'rows': {'trackA': {'모델': 'Track A · GNN 앙상블 [base]',
   'Val PR-AUC': 0.6364887622071024,
   'Test PR-AUC': 0.5808846620823196,
   '패턴내R@예산': 90.2046783625731,
   '패턴밖R@예산': 1.9607843137254901,
   '시나리오R%': 93.98907103825137,
   'P@예산': 61.61417322834646,
   'F1@예산': 57.98981009726726,
   'val패턴내R': 92.4963924963925,
   'val패턴밖R': 2.570694087403599},
  'trackB': {'모델': 'Track B · LightGBM 1칸 [base]',
   'Val PR-AUC': 0.6273199857673158,
   'Test PR-AUC': 0.5550014955577198,
   '패턴내R@예산': 81.72514619883042,
   '패턴밖R@예산': 5.88235294117647,
   '시나리오R%': 95.08196721311475,
   'P@예산': 57.677165354330704,
   'F1@예산': 54.28439092172302,
   'val패턴내R': 88.